In [1]:
import os
import numpy as np
from datetime import datetime
# import tf2onnx
import keras
from keras.metrics import BinaryAccuracy, SparseCategoricalAccuracy
from keras.models import load_model
from tqdm import tqdm  # For dynamic progress updates
# from cnn.model import ResNetClassifier
from utils import load_features_from_hdf5
from data_preprocessing import (
    process_for_complex_cnn,
    process_for_complex_cnn_tdl_cdl,
    process_for_complex_cnn_channel,
    process_for_complex_cnn_channel_tdl_cdl,
)

from cnn.layers import (
    downsample,
    upsample,
    integrate_oper_upsample,
    integrate_oper_downsample,
    EncoderMiniBlock,
    DecoderMiniBlock,
    Oper2D,
    Oper2DTranspose,
)
# from cnn.model import ResNetClassifier
# from cnn.module import ResNet
from evaluation import print_metrics, print_metrics_multiclass
from visualization import (
    plot_confusion_matrix,
    plot_confusion_matrix_multiclass,
    plot_roc_curve,
    plot_roc_curve_multiclass,
)
# import onnxruntime as ort  # ONNX Runtime for inference
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0  # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
    print(f"Using GPU {gpu_num}")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel("ERROR")

2025-06-20 13:17:08.758155: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-20 13:17:09.784286: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2025-06-20 13:17:09.784384: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer_plugin.so.7'; dlerror: libnvinfer_plugin.so.7: cannot open shared object file: No such file or directory
2025-06-20 13:17:09.784396: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Cannot dlopen some TensorRT libraries. If you would like t

Using GPU 0


In [2]:
# Load features from HDF5 files

mimo_los_test = load_features_from_hdf5('data/dataset_sionna_mimo_los_test.h5')
mimo_los_train = load_features_from_hdf5('data/dataset_sionna_mimo_los.h5')
mimo_nlos_test = load_features_from_hdf5('data/dataset_sionna_mimo_nlos_test.h5')
mimo_nlos_train = load_features_from_hdf5('data/dataset_sionna_mimo_nlos.h5')

siso_los_test = load_features_from_hdf5('data/dataset_sionna_siso_los_test.h5')
siso_los_train = load_features_from_hdf5('data/dataset_sionna_siso_los.h5')
siso_nlos_test = load_features_from_hdf5('data/dataset_sionna_siso_nlos_test.h5')
siso_nlos_train = load_features_from_hdf5('data/dataset_sionna_siso_nlos.h5')

tdl_cdl_test = load_features_from_hdf5('dataset/dataset_tdl_cdl_test.h5')
tdl_cdl_train = load_features_from_hdf5('dataset/dataset_tdl_cdl.h5')

tdl_cdl_train_siso = load_features_from_hdf5('data/dataset_tdl_cdl_siso.h5')
tdl_cdl_test_siso = load_features_from_hdf5('data/dataset_tdl_cdl_test_siso.h5')
tdl_cdl_train_mimo = load_features_from_hdf5('data/dataset_tdl_cdl_mimo.h5')
tdl_cdl_test_mimo = load_features_from_hdf5('data/dataset_tdl_cdl_test_mimo.h5')

tdl_cdl_0db_siso_test = load_features_from_hdf5('data/dataset_tdl_cdl_0db_siso.h5')
tdl_cdl_5db_siso_test = load_features_from_hdf5('data/dataset_tdl_cdl_5db_siso.h5')
tdl_cdl_10db_siso_test = load_features_from_hdf5('data/dataset_tdl_cdl_10db_siso.h5')
tdl_cdl_15db_siso_test = load_features_from_hdf5('data/dataset_tdl_cdl_15db_siso.h5')
tdl_cdl_0db_mimo_test = load_features_from_hdf5('data/dataset_tdl_cdl_0db_mimo.h5')
tdl_cdl_5db_mimo_test = load_features_from_hdf5('data/dataset_tdl_cdl_5db_mimo.h5')
tdl_cdl_10db_mimo_test = load_features_from_hdf5('data/dataset_tdl_cdl_10db_mimo.h5')
tdl_cdl_15db_mimo_test = load_features_from_hdf5('data/dataset_tdl_cdl_15db_mimo.h5')

# Process for complex CNN (replace extract_and_concatenate_with_grouping)
train_mimo_los, train_mimo_los_labels = process_for_complex_cnn(mimo_los_train)
train_mimo_nlos, train_mimo_nlos_labels = process_for_complex_cnn(mimo_nlos_train)
train_siso_los, train_siso_los_labels = process_for_complex_cnn(siso_los_train)
train_siso_nlos, train_siso_nlos_labels = process_for_complex_cnn(siso_nlos_train)

train_siso_los_without_duplicate, train_siso_los_labels_without_duplicate = process_for_complex_cnn(siso_los_train)
train_siso_nlos_without_duplicate, train_siso_nlos_labels_without_duplicate = process_for_complex_cnn(siso_nlos_train)

test_mimo_los, test_mimo_los_labels = process_for_complex_cnn(mimo_los_test)
test_mimo_nlos, test_mimo_nlos_labels = process_for_complex_cnn(mimo_nlos_test)
test_siso_los, test_siso_los_labels = process_for_complex_cnn(siso_los_test)
test_siso_nlos, test_siso_nlos_labels = process_for_complex_cnn(siso_nlos_test)

# train_tdls, train_cdls, train_tdls_labels, train_cdls_labels = process_for_complex_cnn_tdl_cdl(tdl_cdl_train)
# test_tdls, test_cdls, test_tdls_labels, test_cdls_labels = process_for_complex_cnn_tdl_cdl(tdl_cdl_test)

train_tdls_siso, train_cdls_siso, train_tdls_labels_siso, train_cdls_labels_siso = process_for_complex_cnn_tdl_cdl(tdl_cdl_train_siso)
test_tdls_siso, test_cdls_siso, test_tdls_labels_siso, test_cdls_labels_siso = process_for_complex_cnn_tdl_cdl(tdl_cdl_test_siso)
train_tdls_mimo, train_cdls_mimo, train_tdls_labels_mimo, train_cdls_labels_mimo = process_for_complex_cnn_tdl_cdl(tdl_cdl_train_mimo)
test_tdls_mimo, test_cdls_mimo, test_tdls_labels_mimo, test_cdls_labels_mimo = process_for_complex_cnn_tdl_cdl(tdl_cdl_test_mimo)

test_tdls_0db_siso, test_cdls_0db_siso, test_tdls_labels_0db_siso, test_cdls_labels_0db_siso = process_for_complex_cnn_tdl_cdl(tdl_cdl_0db_siso_test)
test_tdls_5db_siso, test_cdls_5db_siso, test_tdls_labels_5db_siso, test_cdls_labels_5db_siso = process_for_complex_cnn_tdl_cdl(tdl_cdl_5db_siso_test)
test_tdls_10db_siso, test_cdls_10db_siso, test_tdls_labels_10db_siso, test_cdls_labels_10db_siso = process_for_complex_cnn_tdl_cdl(tdl_cdl_10db_siso_test)
test_tdls_15db_siso, test_cdls_15db_siso, test_tdls_labels_15db_siso, test_cdls_labels_15db_siso = process_for_complex_cnn_tdl_cdl(tdl_cdl_15db_siso_test)
test_tdls_0db_mimo, test_cdls_0db_mimo, test_tdls_labels_0db_mimo, test_cdls_labels_0db_mimo = process_for_complex_cnn_tdl_cdl(tdl_cdl_0db_mimo_test)
test_tdls_5db_mimo, test_cdls_5db_mimo, test_tdls_labels_5db_mimo, test_cdls_labels_5db_mimo = process_for_complex_cnn_tdl_cdl(tdl_cdl_5db_mimo_test)
test_tdls_10db_mimo, test_cdls_10db_mimo, test_tdls_labels_10db_mimo, test_cdls_labels_10db_mimo = process_for_complex_cnn_tdl_cdl(tdl_cdl_10db_mimo_test)
test_tdls_15db_mimo, test_cdls_15db_mimo, test_tdls_labels_15db_mimo, test_cdls_labels_15db_mimo = process_for_complex_cnn_tdl_cdl(tdl_cdl_15db_mimo_test)

# If you have process_for_complex_cnn_channel and process_for_complex_cnn_channel_tdl_cdl, use them for channel data
train_mimo_los_channel, train_mimo_los_labels_channel = process_for_complex_cnn_channel(mimo_los_train)
train_mimo_nlos_channel, train_mimo_nlos_labels_channel = process_for_complex_cnn_channel(mimo_nlos_train)
train_siso_los_channel, train_siso_los_labels_channel = process_for_complex_cnn_channel(siso_los_train)
train_siso_nlos_channel, train_siso_nlos_labels_channel = process_for_complex_cnn_channel(siso_nlos_train)

train_siso_los_channel_without_duplicate, train_siso_los_labels_channel_without_duplicate = process_for_complex_cnn_channel(siso_los_train)
train_siso_nlos_channel_without_duplicate, train_siso_nlos_labels_channel_without_duplicate = process_for_complex_cnn_channel(siso_nlos_train)
test_mimo_los_channel, test_mimo_los_labels_channel = process_for_complex_cnn_channel(mimo_los_test)
test_mimo_nlos_channel, test_mimo_nlos_labels_channel = process_for_complex_cnn_channel(mimo_nlos_test)
test_siso_los_channel, test_siso_los_labels_channel = process_for_complex_cnn_channel(siso_los_test)
test_siso_nlos_channel, test_siso_nlos_labels_channel = process_for_complex_cnn_channel(siso_nlos_test)

# train_tdls_channel, train_cdls_channel, label_tdls_channel, label_cdls_channel = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_train)
# test_tdls_channel, test_cdls_channel, test_label_tdls_channel, test_label_cdls_channel = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_test)

train_tdls_channel_siso, train_cdls_channel_siso, label_tdls_channel_siso, label_cdls_channel_siso = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_train_siso)
test_tdls_channel_siso, test_cdls_channel_siso, test_label_tdls_channel_siso, test_label_cdls_channel_siso = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_test_siso)
train_tdls_channel_mimo, train_cdls_channel_mimo, label_tdls_channel_mimo, label_cdls_channel_mimo = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_train_mimo)
test_tdls_channel_mimo, test_cdls_channel_mimo, test_label_tdls_channel_mimo, test_label_cdls_channel_mimo = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_test_mimo)

test_tdls_channel_0db_siso, test_cdls_channel_0db_siso, test_label_tdl_channel_0db_siso, test_label_cdl_channel_0db_siso = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_0db_siso_test)
test_tdls_channel_5db_siso, test_cdls_channel_5db_siso, test_label_tdl_channel_5db_siso, test_label_cdl_channel_5db_siso = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_5db_siso_test)
test_tdls_channel_10db_siso, test_cdls_channel_10db_siso, test_label_tdl_channel_10db_siso, test_label_cdl_channel_10db_siso = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_10db_siso_test)
test_tdls_channel_15db_siso, test_cdls_channel_15db_siso, test_label_tdl_channel_15db_siso, test_label_cdl_channel_15db_siso = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_15db_siso_test)
test_tdls_channel_0db_mimo, test_cdls_channel_0db_mimo, test_label_tdl_channel_0db_mimo, test_label_cdl_channel_0db_mimo = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_0db_mimo_test)
test_tdls_channel_5db_mimo, test_cdls_channel_5db_mimo, test_label_tdl_channel_5db_mimo, test_label_cdl_channel_5db_mimo = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_5db_mimo_test)
test_tdls_channel_10db_mimo, test_cdls_channel_10db_mimo, test_label_tdl_channel_10db_mimo, test_label_cdl_channel_10db_mimo = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_10db_mimo_test)
test_tdls_channel_15db_mimo, test_cdls_channel_15db_mimo, test_label_tdl_channel_15db_mimo, test_label_cdl_channel_15db_mimo = process_for_complex_cnn_channel_tdl_cdl(tdl_cdl_15db_mimo_test)

In [3]:
# print("train_tdls shape:", train_tdls.shape)
# print("train_cdls shape:", train_cdls.shape)
# print("train_tdls_channel shape:", train_tdls_channel.shape)
# print("train_cdls_channel shape:", train_cdls_channel.shape)
# print("train_tdls_labels shape:", train_tdls_labels.shape)
# print("train_cdls_labels shape:", train_cdls_labels.shape)
# print("test_tdls shape:", test_tdls.shape)
# print("test_cdls shape:", test_cdls.shape)
# print("test_tdls_channel shape:", test_tdls_channel.shape)
# print("test_cdls_channel shape:", test_cdls_channel.shape)
# print("test_tdls_labels shape:", test_tdls_labels.shape)
# print("test_cdls_labels shape:", test_cdls_labels.shape)
print("train_tdls_labels_siso shape:", train_tdls_labels_siso.shape)
print("train_cdls_labels_siso shape:", train_cdls_labels_siso.shape)
print("train_tdls_labels_mimo shape:", train_tdls_labels_mimo.shape)
print("train_cdls_labels_mimo shape:", train_cdls_labels_mimo.shape)
print("test_tdls_labels_siso shape:", test_tdls_labels_siso.shape)
print("test_cdls_labels_siso shape:", test_cdls_labels_siso.shape)
print("test_tdls_labels_mimo shape:", test_tdls_labels_mimo.shape)
print("test_cdls_labels_mimo shape:", test_cdls_labels_mimo.shape)
print("train_tdls_labels_0db_siso shape:", test_tdls_channel_0db_siso.shape)



train_tdls_labels_siso shape: (10000,)
train_cdls_labels_siso shape: (10000,)
train_tdls_labels_mimo shape: (10000,)
train_cdls_labels_mimo shape: (10000,)
test_tdls_labels_siso shape: (5000,)
test_cdls_labels_siso shape: (5000,)
test_tdls_labels_mimo shape: (5000,)
test_cdls_labels_mimo shape: (5000,)
train_tdls_labels_0db_siso shape: (5000, 1, 1, 1, 14, 192)


In [4]:
# Combine dimensions for MIMO data (e.g., 1 and 2 -> 1 * 2 = 2)
train_mimo_los = train_mimo_los.reshape(train_mimo_los.shape[0], train_mimo_los.shape[1], train_mimo_los.shape[2] * train_mimo_los.shape[3], train_mimo_los.shape[4], train_mimo_los.shape[5])
train_mimo_nlos = train_mimo_nlos.reshape(train_mimo_nlos.shape[0], train_mimo_nlos.shape[1], train_mimo_nlos.shape[2] * train_mimo_nlos.shape[3], train_mimo_nlos.shape[4], train_mimo_nlos.shape[5])
test_mimo_los = test_mimo_los.reshape(test_mimo_los.shape[0], test_mimo_los.shape[1], test_mimo_los.shape[2] * test_mimo_los.shape[3], test_mimo_los.shape[4], test_mimo_los.shape[5])
test_mimo_nlos = test_mimo_nlos.reshape(test_mimo_nlos.shape[0], test_mimo_nlos.shape[1], test_mimo_nlos.shape[2] * test_mimo_nlos.shape[3], test_mimo_nlos.shape[4], test_mimo_nlos.shape[5])
print("train_mimo_los shape after reshape:", train_mimo_los.shape)
print("train_mimo_nlos shape after reshape:", train_mimo_nlos.shape)
print("test_mimo_los shape after reshape:", test_mimo_los.shape)
print("test_mimo_nlos shape after reshape:", test_mimo_nlos.shape)

train_mimo_los_channel = train_mimo_los_channel.reshape(train_mimo_los_channel.shape[0], train_mimo_los_channel.shape[1], train_mimo_los_channel.shape[2] * train_mimo_los_channel.shape[3], train_mimo_los_channel.shape[4], train_mimo_los_channel.shape[5])
train_mimo_nlos_channel = train_mimo_nlos_channel.reshape(train_mimo_nlos_channel.shape[0], train_mimo_nlos_channel.shape[1], train_mimo_nlos_channel.shape[2] * train_mimo_nlos_channel.shape[3], train_mimo_nlos_channel.shape[4], train_mimo_nlos_channel.shape[5])
test_mimo_los_channel = test_mimo_los_channel.reshape(test_mimo_los_channel.shape[0], test_mimo_los_channel.shape[1], test_mimo_los_channel.shape[2] * test_mimo_los_channel.shape[3], test_mimo_los_channel.shape[4], test_mimo_los_channel.shape[5])
test_mimo_nlos_channel = test_mimo_nlos_channel.reshape(test_mimo_nlos_channel.shape[0], test_mimo_nlos_channel.shape[1], test_mimo_nlos_channel.shape[2] * test_mimo_nlos_channel.shape[3], test_mimo_nlos_channel.shape[4], test_mimo_nlos_channel.shape[5])
print("train_mimo_los_channel shape after reshape:", train_mimo_los_channel.shape)
print("train_mimo_nlos_channel shape after reshape:", train_mimo_nlos_channel.shape)
print("test_mimo_los_channel shape after reshape:", test_mimo_los_channel.shape)
print("test_mimo_nlos_channel shape after reshape:", test_mimo_nlos_channel.shape)

# Combine dimensions for SISO data (e.g., 1 and 2 -> 1 * 2 = 2)
train_siso_los = train_siso_los.reshape(train_siso_los.shape[0], train_siso_los.shape[1], train_siso_los.shape[2] * train_siso_los.shape[3], train_siso_los.shape[4], train_siso_los.shape[5])
train_siso_nlos = train_siso_nlos.reshape(train_siso_nlos.shape[0], train_siso_nlos.shape[1], train_siso_nlos.shape[2] * train_siso_nlos.shape[3], train_siso_nlos.shape[4], train_siso_nlos.shape[5])
test_siso_los = test_siso_los.reshape(test_siso_los.shape[0], test_siso_los.shape[1], test_siso_los.shape[2] * test_siso_los.shape[3], test_siso_los.shape[4], test_siso_los.shape[5])
test_siso_nlos = test_siso_nlos.reshape(test_siso_nlos.shape[0], test_siso_nlos.shape[1], test_siso_nlos.shape[2] * test_siso_nlos.shape[3], test_siso_nlos.shape[4], test_siso_nlos.shape[5])
print("train_siso_los shape after reshape:", train_siso_los.shape)
print("train_siso_nlos shape after reshape:", train_siso_nlos.shape)
print("test_siso_los shape after reshape:", test_siso_los.shape)
print("test_siso_nlos shape after reshape:", test_siso_nlos.shape)

train_siso_los_channel = train_siso_los_channel.reshape(train_siso_los_channel.shape[0], train_siso_los_channel.shape[1], train_siso_los_channel.shape[2] * train_siso_los_channel.shape[3], train_siso_los_channel.shape[4], train_siso_los_channel.shape[5])
train_siso_nlos_channel = train_siso_nlos_channel.reshape(train_siso_nlos_channel.shape[0], train_siso_nlos_channel.shape[1], train_siso_nlos_channel.shape[2] * train_siso_nlos_channel.shape[3], train_siso_nlos_channel.shape[4], train_siso_nlos_channel.shape[5])
test_siso_los_channel = test_siso_los_channel.reshape(test_siso_los_channel.shape[0], test_siso_los_channel.shape[1], test_siso_los_channel.shape[2] * test_siso_los_channel.shape[3], test_siso_los_channel.shape[4], test_siso_los_channel.shape[5])
test_siso_nlos_channel = test_siso_nlos_channel.reshape(test_siso_nlos_channel.shape[0], test_siso_nlos_channel.shape[1], test_siso_nlos_channel.shape[2] * test_siso_nlos_channel.shape[3], test_siso_nlos_channel.shape[4], test_siso_nlos_channel.shape[5])
print("train_siso_los_channel shape after reshape:", train_siso_los_channel.shape)
print("train_siso_nlos_channel shape after reshape:", train_siso_nlos_channel.shape)
print("test_siso_los_channel shape after reshape:", test_siso_los_channel.shape)
print("test_siso_nlos_channel shape after reshape:", test_siso_nlos_channel.shape)

# Combine dimensions for TDL/CDL data (e.g., 1 and 1 -> 1 * 1 = 1, effectively removing them)
# train_tdls = train_tdls.reshape(train_tdls.shape[0], train_tdls.shape[1], train_tdls.shape[2] * train_tdls.shape[3], train_tdls.shape[4], train_tdls.shape[5])
# train_cdls = train_cdls.reshape(train_cdls.shape[0], train_cdls.shape[1], train_cdls.shape[2] * train_cdls.shape[3], train_cdls.shape[4], train_cdls.shape[5])
# print("train_tdls shape after reshape:", train_tdls.shape)
# print("train_cdls shape after reshape:", train_cdls.shape)

# train_tdls_channel = train_tdls_channel.reshape(train_tdls_channel.shape[0], train_tdls_channel.shape[1], train_tdls_channel.shape[2] * train_tdls_channel.shape[3], train_tdls_channel.shape[4], train_tdls_channel.shape[5])
# train_cdls_channel = train_cdls_channel.reshape(train_cdls_channel.shape[0], train_cdls_channel.shape[1], train_cdls_channel.shape[2] * train_cdls_channel.shape[3], train_cdls_channel.shape[4], train_cdls_channel.shape[5])
# print("train_tdls_channel shape after reshape:", train_tdls_channel.shape)
# print("train_cdls_channel shape after reshape:", train_cdls_channel.shape)    

# test_cdls = test_cdls.reshape(test_cdls.shape[0], test_cdls.shape[1], test_cdls.shape[2] * test_cdls.shape[3], test_cdls.shape[4], test_cdls.shape[5])
# test_tdls = test_tdls.reshape(test_tdls.shape[0], test_tdls.shape[1], test_tdls.shape[2] * test_tdls.shape[3], test_tdls.shape[4], test_tdls.shape[5])
# print("test_tdls shape after reshape:", test_tdls.shape)
# print("test_cdls shape after reshape:", test_cdls.shape)
# test_tdls_channel = test_tdls_channel.reshape(test_tdls_channel.shape[0], test_tdls_channel.shape[1], test_tdls_channel.shape[2] * test_tdls_channel.shape[3], test_tdls_channel.shape[4], test_tdls_channel.shape[5])
# test_cdls_channel = test_cdls_channel.reshape(test_cdls_channel.shape[0], test_cdls_channel.shape[1], test_cdls_channel.shape[2] * test_cdls_channel.shape[3], test_cdls_channel.shape[4], test_cdls_channel.shape[5])
# print("test_tdls_channel shape after reshape:", test_tdls_channel.shape)
# print("test_cdls_channel shape after reshape:", test_cdls_channel.shape)

train_tdls_siso = train_tdls_siso.reshape(train_tdls_siso.shape[0], train_tdls_siso.shape[1], train_tdls_siso.shape[2] * train_tdls_siso.shape[3], train_tdls_siso.shape[4], train_tdls_siso.shape[5])
train_cdls_siso = train_cdls_siso.reshape(train_cdls_siso.shape[0], train_cdls_siso.shape[1], train_cdls_siso.shape[2] * train_cdls_siso.shape[3], train_cdls_siso.shape[4], train_cdls_siso.shape[5])
print("train_tdls_siso shape after reshape:", train_tdls_siso.shape)
print("train_cdls_siso shape after reshape:", train_cdls_siso.shape)
test_tdls_siso = test_tdls_siso.reshape(test_tdls_siso.shape[0], test_tdls_siso.shape[1], test_tdls_siso.shape[2] * test_tdls_siso.shape[3], test_tdls_siso.shape[4], test_tdls_siso.shape[5])
test_cdls_siso = test_cdls_siso.reshape(test_cdls_siso.shape[0], test_cdls_siso.shape[1], test_cdls_siso.shape[2] * test_cdls_siso.shape[3], test_cdls_siso.shape[4], test_cdls_siso.shape[5])
print("test_tdls_siso shape after reshape:", test_tdls_siso.shape)
print("test_cdls_siso shape after reshape:", test_cdls_siso.shape)
train_tdls_mimo = train_tdls_mimo.reshape(train_tdls_mimo.shape[0], train_tdls_mimo.shape[1], train_tdls_mimo.shape[2] * train_tdls_mimo.shape[3], train_tdls_mimo.shape[4], train_tdls_mimo.shape[5])
train_cdls_mimo = train_cdls_mimo.reshape(train_cdls_mimo.shape[0], train_cdls_mimo.shape[1], train_cdls_mimo.shape[2] * train_cdls_mimo.shape[3], train_cdls_mimo.shape[4], train_cdls_mimo.shape[5])
print("train_tdls_mimo shape after reshape:", train_tdls_mimo.shape)
print("train_cdls_mimo shape after reshape:", train_cdls_mimo.shape)
test_tdls_mimo = test_tdls_mimo.reshape(test_tdls_mimo.shape[0], test_tdls_mimo.shape[1], test_tdls_mimo.shape[2] * test_tdls_mimo.shape[3], test_tdls_mimo.shape[4], test_tdls_mimo.shape[5])
test_cdls_mimo = test_cdls_mimo.reshape(test_cdls_mimo.shape[0], test_cdls_mimo.shape[1], test_cdls_mimo.shape[2] * test_cdls_mimo.shape[3], test_cdls_mimo.shape[4], test_cdls_mimo.shape[5])
print("test_tdls_mimo shape after reshape:", test_tdls_mimo.shape)
print("test_cdls_mimo shape after reshape:", test_cdls_mimo.shape)

test_tdls_0db_siso = test_tdls_0db_siso.reshape(test_tdls_0db_siso.shape[0], test_tdls_0db_siso.shape[1], test_tdls_0db_siso.shape[2] * test_tdls_0db_siso.shape[3], test_tdls_0db_siso.shape[4], test_tdls_0db_siso.shape[5])
test_cdls_0db_siso = test_cdls_0db_siso.reshape(test_cdls_0db_siso.shape[0], test_cdls_0db_siso.shape[1], test_cdls_0db_siso.shape[2] * test_cdls_0db_siso.shape[3], test_cdls_0db_siso.shape[4], test_cdls_0db_siso.shape[5])
print("test_tdls_0db_siso shape after reshape:", test_tdls_0db_siso.shape)
print("test_cdls_0db_siso shape after reshape:", test_cdls_0db_siso.shape)
test_tdls_5db_siso = test_tdls_5db_siso.reshape(test_tdls_5db_siso.shape[0], test_tdls_5db_siso.shape[1], test_tdls_5db_siso.shape[2] * test_tdls_5db_siso.shape[3], test_tdls_5db_siso.shape[4], test_tdls_5db_siso.shape[5])
test_cdls_5db_siso = test_cdls_5db_siso.reshape(test_cdls_5db_siso.shape[0], test_cdls_5db_siso.shape[1], test_cdls_5db_siso.shape[2] * test_cdls_5db_siso.shape[3], test_cdls_5db_siso.shape[4], test_cdls_5db_siso.shape[5])
print("test_tdls_5db_siso shape after reshape:", test_tdls_5db_siso.shape)
print("test_cdls_5db_siso shape after reshape:", test_cdls_5db_siso.shape)
test_tdls_10db_siso = test_tdls_10db_siso.reshape(test_tdls_10db_siso.shape[0], test_tdls_10db_siso.shape[1], test_tdls_10db_siso.shape[2] * test_tdls_10db_siso.shape[3], test_tdls_10db_siso.shape[4], test_tdls_10db_siso.shape[5])
test_cdls_10db_siso = test_cdls_10db_siso.reshape(test_cdls_10db_siso.shape[0], test_cdls_10db_siso.shape[1], test_cdls_10db_siso.shape[2] * test_cdls_10db_siso.shape[3], test_cdls_10db_siso.shape[4], test_cdls_10db_siso.shape[5])
print("test_tdls_10db_siso shape after reshape:", test_tdls_10db_siso.shape)
print("test_cdls_10db_siso shape after reshape:", test_cdls_10db_siso.shape)
test_tdls_15db_siso = test_tdls_15db_siso.reshape(test_tdls_15db_siso.shape[0], test_tdls_15db_siso.shape[1], test_tdls_15db_siso.shape[2] * test_tdls_15db_siso.shape[3], test_tdls_15db_siso.shape[4], test_tdls_15db_siso.shape[5])
test_cdls_15db_siso = test_cdls_15db_siso.reshape(test_cdls_15db_siso.shape[0], test_cdls_15db_siso.shape[1], test_cdls_15db_siso.shape[2] * test_cdls_15db_siso.shape[3], test_cdls_15db_siso.shape[4], test_cdls_15db_siso.shape[5])
print("test_tdls_15db_siso shape after reshape:", test_tdls_15db_siso.shape)
print("test_cdls_15db_siso shape after reshape:", test_cdls_15db_siso.shape)
test_tdls_0db_mimo = test_tdls_0db_mimo.reshape(test_tdls_0db_mimo.shape[0], test_tdls_0db_mimo.shape[1], test_tdls_0db_mimo.shape[2] * test_tdls_0db_mimo.shape[3], test_tdls_0db_mimo.shape[4], test_tdls_0db_mimo.shape[5])
test_cdls_0db_mimo = test_cdls_0db_mimo.reshape(test_cdls_0db_mimo.shape[0], test_cdls_0db_mimo.shape[1], test_cdls_0db_mimo.shape[2] * test_cdls_0db_mimo.shape[3], test_cdls_0db_mimo.shape[4], test_cdls_0db_mimo.shape[5])
print("test_tdls_0db_mimo shape after reshape:", test_tdls_0db_mimo.shape)
print("test_cdls_0db_mimo shape after reshape:", test_cdls_0db_mimo.shape)
test_tdls_5db_mimo = test_tdls_5db_mimo.reshape(test_tdls_5db_mimo.shape[0], test_tdls_5db_mimo.shape[1], test_tdls_5db_mimo.shape[2] * test_tdls_5db_mimo.shape[3], test_tdls_5db_mimo.shape[4], test_tdls_5db_mimo.shape[5])
test_cdls_5db_mimo = test_cdls_5db_mimo.reshape(test_cdls_5db_mimo.shape[0], test_cdls_5db_mimo.shape[1], test_cdls_5db_mimo.shape[2] * test_cdls_5db_mimo.shape[3], test_cdls_5db_mimo.shape[4], test_cdls_5db_mimo.shape[5])
print("test_tdls_5db_mimo shape after reshape:", test_tdls_5db_mimo.shape)
print("test_cdls_5db_mimo shape after reshape:", test_cdls_5db_mimo.shape)
test_tdls_10db_mimo = test_tdls_10db_mimo.reshape(test_tdls_10db_mimo.shape[0], test_tdls_10db_mimo.shape[1], test_tdls_10db_mimo.shape[2] * test_tdls_10db_mimo.shape[3], test_tdls_10db_mimo.shape[4], test_tdls_10db_mimo.shape[5])
test_cdls_10db_mimo = test_cdls_10db_mimo.reshape(test_cdls_10db_mimo.shape[0], test_cdls_10db_mimo.shape[1], test_cdls_10db_mimo.shape[2] * test_cdls_10db_mimo.shape[3], test_cdls_10db_mimo.shape[4], test_cdls_10db_mimo.shape[5])
print("test_tdls_10db_mimo shape after reshape:", test_tdls_10db_mimo.shape)
print("test_cdls_10db_mimo shape after reshape:", test_cdls_10db_mimo.shape)
test_tdls_15db_mimo = test_tdls_15db_mimo.reshape(test_tdls_15db_mimo.shape[0], test_tdls_15db_mimo.shape[1], test_tdls_15db_mimo.shape[2] * test_tdls_15db_mimo.shape[3], test_tdls_15db_mimo.shape[4], test_tdls_15db_mimo.shape[5])
test_cdls_15db_mimo = test_cdls_15db_mimo.reshape(test_cdls_15db_mimo.shape[0], test_cdls_15db_mimo.shape[1], test_cdls_15db_mimo.shape[2] * test_cdls_15db_mimo.shape[3], test_cdls_15db_mimo.shape[4], test_cdls_15db_mimo.shape[5])
print("test_tdls_15db_mimo shape after reshape:", test_tdls_15db_mimo.shape)
print("test_cdls_15db_mimo shape after reshape:", test_cdls_15db_mimo.shape)


test_tdls_channel_0db_siso = test_tdls_channel_0db_siso.reshape(test_tdls_channel_0db_siso.shape[0], test_tdls_channel_0db_siso.shape[1], test_tdls_channel_0db_siso.shape[2] * test_tdls_channel_0db_siso.shape[3], test_tdls_channel_0db_siso.shape[4], test_tdls_channel_0db_siso.shape[5])
test_cdls_channel_0db_siso = test_cdls_channel_0db_siso.reshape(test_cdls_channel_0db_siso.shape[0], test_cdls_channel_0db_siso.shape[1], test_cdls_channel_0db_siso.shape[2] * test_cdls_channel_0db_siso.shape[3], test_cdls_channel_0db_siso.shape[4], test_cdls_channel_0db_siso.shape[5])
print("test_tdls_channel_0db_siso shape after reshape:", test_tdls_channel_0db_siso.shape)
print("test_cdls_channel_0db_siso shape after reshape:", test_cdls_channel_0db_siso.shape)
test_tdls_channel_5db_siso = test_tdls_channel_5db_siso.reshape(test_tdls_channel_5db_siso.shape[0], test_tdls_channel_5db_siso.shape[1], test_tdls_channel_5db_siso.shape[2] * test_tdls_channel_5db_siso.shape[3], test_tdls_channel_5db_siso.shape[4], test_tdls_channel_5db_siso.shape[5])
test_cdls_channel_5db_siso = test_cdls_channel_5db_siso.reshape(test_cdls_channel_5db_siso.shape[0], test_cdls_channel_5db_siso.shape[1], test_cdls_channel_5db_siso.shape[2] * test_cdls_channel_5db_siso.shape[3], test_cdls_channel_5db_siso.shape[4], test_cdls_channel_5db_siso.shape[5])
print("test_tdls_channel_5db_siso shape after reshape:", test_tdls_channel_5db_siso.shape)
print("test_cdls_channel_5db_siso shape after reshape:", test_cdls_channel_5db_siso.shape)
test_tdls_channel_10db_siso = test_tdls_channel_10db_siso.reshape(test_tdls_channel_10db_siso.shape[0], test_tdls_channel_10db_siso.shape[1], test_tdls_channel_10db_siso.shape[2] * test_tdls_channel_10db_siso.shape[3], test_tdls_channel_10db_siso.shape[4], test_tdls_channel_10db_siso.shape[5])
test_cdls_channel_10db_siso = test_cdls_channel_10db_siso.reshape(test_cdls_channel_10db_siso.shape[0], test_cdls_channel_10db_siso.shape[1], test_cdls_channel_10db_siso.shape[2] * test_cdls_channel_10db_siso.shape[3], test_cdls_channel_10db_siso.shape[4], test_cdls_channel_10db_siso.shape[5])
print("test_tdls_channel_10db_siso shape after reshape:", test_tdls_channel_10db_siso.shape)
print("test_cdls_channel_10db_siso shape after reshape:", test_cdls_channel_10db_siso.shape)
test_tdls_channel_15db_siso = test_tdls_channel_15db_siso.reshape(test_tdls_channel_15db_siso.shape[0], test_tdls_channel_15db_siso.shape[1], test_tdls_channel_15db_siso.shape[2] * test_tdls_channel_15db_siso.shape[3], test_tdls_channel_15db_siso.shape[4], test_tdls_channel_15db_siso.shape[5])
test_cdls_channel_15db_siso = test_cdls_channel_15db_siso.reshape(test_cdls_channel_15db_siso.shape[0], test_cdls_channel_15db_siso.shape[1], test_cdls_channel_15db_siso.shape[2] * test_cdls_channel_15db_siso.shape[3], test_cdls_channel_15db_siso.shape[4], test_cdls_channel_15db_siso.shape[5])
print("test_tdls_channel_15db_siso shape after reshape:", test_tdls_channel_15db_siso.shape)
print("test_cdls_channel_15db_siso shape after reshape:", test_cdls_channel_15db_siso.shape)
test_tdls_channel_0db_mimo = test_tdls_channel_0db_mimo.reshape(test_tdls_channel_0db_mimo.shape[0], test_tdls_channel_0db_mimo.shape[1], test_tdls_channel_0db_mimo.shape[2] * test_tdls_channel_0db_mimo.shape[3], test_tdls_channel_0db_mimo.shape[4], test_tdls_channel_0db_mimo.shape[5])
test_cdls_channel_0db_mimo = test_cdls_channel_0db_mimo.reshape(test_cdls_channel_0db_mimo.shape[0], test_cdls_channel_0db_mimo.shape[1], test_cdls_channel_0db_mimo.shape[2] * test_cdls_channel_0db_mimo.shape[3], test_cdls_channel_0db_mimo.shape[4], test_cdls_channel_0db_mimo.shape[5])
print("test_tdls_channel_0db_mimo shape after reshape:", test_tdls_channel_0db_mimo.shape)
print("test_cdls_channel_0db_mimo shape after reshape:", test_cdls_channel_0db_mimo.shape)
test_tdls_channel_5db_mimo = test_tdls_channel_5db_mimo.reshape(test_tdls_channel_5db_mimo.shape[0], test_tdls_channel_5db_mimo.shape[1], test_tdls_channel_5db_mimo.shape[2] * test_tdls_channel_5db_mimo.shape[3], test_tdls_channel_5db_mimo.shape[4], test_tdls_channel_5db_mimo.shape[5])
test_cdls_channel_5db_mimo = test_cdls_channel_5db_mimo.reshape(test_cdls_channel_5db_mimo.shape[0], test_cdls_channel_5db_mimo.shape[1], test_cdls_channel_5db_mimo.shape[2] * test_cdls_channel_5db_mimo.shape[3], test_cdls_channel_5db_mimo.shape[4], test_cdls_channel_5db_mimo.shape[5])
print("test_tdls_channel_5db_mimo shape after reshape:", test_tdls_channel_5db_mimo.shape)
print("test_cdls_channel_5db_mimo shape after reshape:", test_cdls_channel_5db_mimo.shape)
test_tdls_channel_10db_mimo = test_tdls_channel_10db_mimo.reshape(test_tdls_channel_10db_mimo.shape[0], test_tdls_channel_10db_mimo.shape[1], test_tdls_channel_10db_mimo.shape[2] * test_tdls_channel_10db_mimo.shape[3], test_tdls_channel_10db_mimo.shape[4], test_tdls_channel_10db_mimo.shape[5])
test_cdls_channel_10db_mimo = test_cdls_channel_10db_mimo.reshape(test_cdls_channel_10db_mimo.shape[0], test_cdls_channel_10db_mimo.shape[1], test_cdls_channel_10db_mimo.shape[2] * test_cdls_channel_10db_mimo.shape[3], test_cdls_channel_10db_mimo.shape[4], test_cdls_channel_10db_mimo.shape[5])
print("test_tdls_channel_10db_mimo shape after reshape:", test_tdls_channel_10db_mimo.shape)
print("test_cdls_channel_10db_mimo shape after reshape:", test_cdls_channel_10db_mimo.shape)
test_tdls_channel_15db_mimo = test_tdls_channel_15db_mimo.reshape(test_tdls_channel_15db_mimo.shape[0], test_tdls_channel_15db_mimo.shape[1], test_tdls_channel_15db_mimo.shape[2] * test_tdls_channel_15db_mimo.shape[3], test_tdls_channel_15db_mimo.shape[4], test_tdls_channel_15db_mimo.shape[5])
test_cdls_channel_15db_mimo = test_cdls_channel_15db_mimo.reshape(test_cdls_channel_15db_mimo.shape[0], test_cdls_channel_15db_mimo.shape[1], test_cdls_channel_15db_mimo.shape[2] * test_cdls_channel_15db_mimo.shape[3], test_cdls_channel_15db_mimo.shape[4], test_cdls_channel_15db_mimo.shape[5])
print("test_tdls_channel_15db_mimo shape after reshape:", test_tdls_channel_15db_mimo.shape)
print("test_cdls_channel_15db_mimo shape after reshape:", test_cdls_channel_15db_mimo.shape)


train_mimo_los shape after reshape: (2000, 1, 2, 14, 192)
train_mimo_nlos shape after reshape: (2000, 1, 2, 14, 192)
test_mimo_los shape after reshape: (1000, 1, 2, 14, 192)
test_mimo_nlos shape after reshape: (1000, 1, 2, 14, 192)
train_mimo_los_channel shape after reshape: (2000, 1, 2, 14, 192)
train_mimo_nlos_channel shape after reshape: (2000, 1, 2, 14, 192)
test_mimo_los_channel shape after reshape: (1000, 1, 2, 14, 192)
test_mimo_nlos_channel shape after reshape: (1000, 1, 2, 14, 192)
train_siso_los shape after reshape: (2000, 1, 1, 14, 192)
train_siso_nlos shape after reshape: (2000, 1, 1, 14, 192)
test_siso_los shape after reshape: (1000, 1, 1, 14, 192)
test_siso_nlos shape after reshape: (1000, 1, 1, 14, 192)
train_siso_los_channel shape after reshape: (2000, 1, 1, 14, 192)
train_siso_nlos_channel shape after reshape: (2000, 1, 1, 14, 192)
test_siso_los_channel shape after reshape: (1000, 1, 1, 14, 192)
test_siso_nlos_channel shape after reshape: (1000, 1, 1, 14, 192)
train_td

In [5]:
# train_data = np.concatenate((train_tdls, train_cdls), axis=0)
# train_data_labels = np.concatenate((train_tdls_labels, train_cdls_labels), axis=0)
# valid_data = np.concatenate((train_mimo_los, train_mimo_nlos, train_siso_los, train_siso_nlos), axis=0)
# valid_data_labels = np.concatenate((train_mimo_los_labels, train_mimo_nlos_labels, train_siso_los_labels, train_siso_nlos_labels), axis=0)
# test_data = np.concatenate((test_tdls, test_cdls), axis=0)
# test_data_labels = np.concatenate((test_tdls_labels, test_cdls_labels), axis=0)
# print("train_data shape:", train_data.shape)
# print("train_labels shape:", train_data_labels.shape)
# print("valid_data shape:", valid_data.shape)
# print("valid_labels shape:", valid_data_labels.shape)
# print("test_data shape:", test_data.shape)
# print("test_labels shape:", test_data_labels.shape)

# train_data_channel = np.concatenate((train_tdls_channel, train_cdls_channel), axis=0)
# train_data_channel_labels = np.concatenate((label_tdls_channel, label_cdls_channel), axis=0)
# valid_data_channel = np.concatenate((train_mimo_los_channel, train_mimo_nlos_channel, train_siso_los_channel, train_siso_nlos_channel), axis=0)
# valid_data_channel_labels = np.concatenate((train_mimo_los_labels_channel, train_mimo_nlos_labels_channel, train_siso_los_labels_channel, train_siso_nlos_labels_channel), axis=0)
# test_data_channel = np.concatenate((test_tdls_channel, test_cdls_channel), axis=0)
# test_data_channel_labels = np.concatenate((test_label_tdls_channel, test_label_cdls_channel), axis=0)
# print("train_data_channel shape:", train_data_channel.shape)
# print("test_data_channel shape:", test_data_channel.shape)
# print("valid_data_channel shape:", valid_data_channel.shape)
# print("train_labels shape:", train_data_labels.shape)
# print("test_labels shape:", test_data_labels.shape)

train_data_siso = np.concatenate((train_tdls_siso, train_cdls_siso), axis=0)
train_data_siso_labels = np.concatenate((train_tdls_labels_siso, train_cdls_labels_siso), axis=0)
# valid_data_siso = np.concatenate((train_mimo_los_channel, train_mimo_nlos_channel, train_siso_los_channel, train_siso_nlos_channel), axis=0)
# valid_data_siso_labels = np.concatenate((train_mimo_los_labels_channel, train_mimo_nlos_labels_channel, train_siso_los_labels_channel, train_siso_nlos_labels_channel), axis=0)
# use the without_duplicate data for validation
valid_data_siso = np.concatenate((train_siso_los_channel_without_duplicate, train_siso_nlos_channel_without_duplicate), axis=0)
valid_data_siso_labels = np.concatenate((train_siso_los_labels_channel_without_duplicate, train_siso_nlos_labels_channel_without_duplicate), axis=0)
test_data_siso = np.concatenate((test_tdls_channel_siso, test_cdls_channel_siso), axis=0)
test_data_siso_labels = np.concatenate((test_tdls_labels_siso, test_cdls_labels_siso), axis=0)
print("train_data_siso shape:", train_data_siso.shape)
print("train_data_siso_labels shape:", train_data_siso_labels.shape)

train_data_mimo = np.concatenate((train_tdls_mimo, train_cdls_mimo), axis=0)
train_data_mimo_labels = np.concatenate((train_tdls_labels_mimo, train_cdls_labels_mimo), axis=0)
valid_data_mimo = np.concatenate((train_mimo_los_channel, train_mimo_nlos_channel), axis=0)
valid_data_mimo_labels = np.concatenate((train_mimo_los_labels_channel, train_mimo_nlos_labels_channel), axis=0)
test_data_mimo = np.concatenate((test_tdls_mimo, test_cdls_mimo), axis=0)
test_data_mimo_labels = np.concatenate((test_tdls_labels_mimo, test_cdls_labels_mimo), axis=0)
print("train_data_mimo shape:", train_data_mimo.shape)
print("train_data_mimo_labels shape:", train_data_mimo_labels.shape)
print("valid_data_mimo shape:", valid_data_mimo.shape)
print("valid_data_mimo_labels shape:", valid_data_mimo_labels.shape)
print("test_data_mimo shape:", test_data_mimo.shape)
print("test_data_mimo_labels shape:", test_data_mimo_labels.shape)

train_data_siso_channel = np.concatenate((train_tdls_channel_siso, train_cdls_channel_siso), axis=0)
train_data_siso_channel_labels = np.concatenate((label_tdls_channel_siso, label_cdls_channel_siso), axis=0)
# valid_data_siso_channel = np.concatenate((train_siso_los_channel, train_siso_nlos_channel), axis=0)
# valid_data_siso_channel_labels = np.concatenate((train_siso_los_labels_channel, train_siso_nlos_labels_channel), axis=0)
test_data_siso_channel = np.concatenate((test_tdls_channel_siso, test_cdls_channel_siso), axis=0)
test_data_siso_channel_labels = np.concatenate((test_label_tdls_channel_siso, test_label_cdls_channel_siso), axis=0)
print("train_data_siso_channel shape:", train_data_siso_channel.shape)
print("train_data_siso_channel_labels shape:", train_data_siso_channel_labels.shape)
# print("valid_data_siso_channel shape:", valid_data_siso_channel.shape)
# print("valid_data_siso_channel_labels shape:", valid_data_siso_channel_labels.shape)
print("test_data_siso_channel shape:", test_data_siso_channel.shape)
print("test_data_siso_channel_labels shape:", test_data_siso_channel_labels.shape)

train_data_mimo_channel = np.concatenate((train_tdls_channel_mimo, train_cdls_channel_mimo), axis=0)
train_data_mimo_channel_labels = np.concatenate((label_tdls_channel_mimo, label_cdls_channel_mimo), axis=0)
# valid_data_mimo_channel = np.concatenate((train_mimo_los_channel, train_mimo_nlos_channel), axis=0)
# valid_data_mimo_channel_labels = np.concatenate((train_mimo_los_labels_channel, train_mimo_nlos_labels_channel), axis=0)
test_data_mimo_channel = np.concatenate((test_tdls_channel_mimo, test_cdls_channel_mimo), axis=0)
test_data_mimo_channel_labels = np.concatenate((test_label_tdls_channel_mimo, test_label_cdls_channel_mimo), axis=0)
print("train_data_mimo_channel shape:", train_data_mimo_channel.shape)
print("train_data_mimo_channel_labels shape:", train_data_mimo_channel_labels.shape)
# print("valid_data_mimo_channel shape:", valid_data_mimo_channel.shape)
# print("valid_data_mimo_channel_labels shape:", valid_data_mimo_channel_labels.shape)
print("test_data_mimo_channel shape:", test_data_mimo_channel.shape)
print("test_data_mimo_channel_labels shape:", test_data_mimo_channel_labels.shape)

test_data_0db_siso = np.concatenate((test_tdls_0db_siso, test_cdls_0db_siso), axis=0)
test_data_0db_siso_labels = np.concatenate((test_tdls_labels_0db_siso, test_cdls_labels_0db_siso), axis=0)
test_data_5db_siso = np.concatenate((test_tdls_5db_siso, test_cdls_5db_siso), axis=0)
test_data_5db_siso_labels = np.concatenate((test_tdls_labels_5db_siso, test_cdls_labels_5db_siso), axis=0)
test_data_10db_siso = np.concatenate((test_tdls_10db_siso, test_cdls_10db_siso), axis=0)
test_data_10db_siso_labels = np.concatenate((test_tdls_labels_10db_siso, test_cdls_labels_10db_siso), axis=0)
test_data_15db_siso = np.concatenate((test_tdls_15db_siso, test_cdls_15db_siso), axis=0)
test_data_15db_siso_labels = np.concatenate((test_tdls_labels_15db_siso, test_cdls_labels_15db_siso), axis=0)
print("test_data_0db_siso shape:", test_data_0db_siso.shape)
print("test_data_0db_siso_labels shape:", test_data_0db_siso_labels.shape)
print("test_data_5db_siso shape:", test_data_5db_siso.shape)
print("test_data_5db_siso_labels shape:", test_data_5db_siso_labels.shape)
print("test_data_10db_siso shape:", test_data_10db_siso.shape)
print("test_data_10db_siso_labels shape:", test_data_10db_siso_labels.shape)
print("test_data_15db_siso shape:", test_data_15db_siso.shape)
print("test_data_15db_siso_labels shape:", test_data_15db_siso_labels.shape)
test_data_0db_mimo = np.concatenate((test_tdls_0db_mimo, test_cdls_0db_mimo), axis=0)
test_data_0db_mimo_labels = np.concatenate((test_tdls_labels_0db_mimo, test_cdls_labels_0db_mimo), axis=0)
test_data_5db_mimo = np.concatenate((test_tdls_5db_mimo, test_cdls_5db_mimo), axis=0)
test_data_5db_mimo_labels = np.concatenate((test_tdls_labels_5db_mimo, test_cdls_labels_5db_mimo), axis=0)
test_data_10db_mimo = np.concatenate((test_tdls_10db_mimo, test_cdls_10db_mimo), axis=0)
test_data_10db_mimo_labels = np.concatenate((test_tdls_labels_10db_mimo, test_cdls_labels_10db_mimo), axis=0)
test_data_15db_mimo = np.concatenate((test_tdls_15db_mimo, test_cdls_15db_mimo), axis=0)
test_data_15db_mimo_labels = np.concatenate((test_tdls_labels_15db_mimo, test_cdls_labels_15db_mimo), axis=0)
print("test_data_0db_mimo shape:", test_data_0db_mimo.shape)
print("test_data_0db_mimo_labels shape:", test_data_0db_mimo_labels.shape)
print("test_data_5db_mimo shape:", test_data_5db_mimo.shape)
print("test_data_5db_mimo_labels shape:", test_data_5db_mimo_labels.shape)
print("test_data_10db_mimo shape:", test_data_10db_mimo.shape)
print("test_data_10db_mimo_labels shape:", test_data_10db_mimo_labels.shape)
print("test_data_15db_mimo shape:", test_data_15db_mimo.shape)
print("test_data_15db_mimo_labels shape:", test_data_15db_mimo_labels.shape)

test_data_channel_0db_siso = np.concatenate((test_tdls_channel_0db_siso, test_cdls_channel_0db_siso), axis=0)
test_data_channel_0db_siso_labels = np.concatenate((test_label_tdl_channel_0db_siso, test_label_cdl_channel_0db_siso), axis=0)
test_data_channel_5db_siso = np.concatenate((test_tdls_channel_5db_siso, test_cdls_channel_5db_siso), axis=0)
test_data_channel_5db_siso_labels = np.concatenate((test_label_tdl_channel_5db_siso, test_label_cdl_channel_5db_siso), axis=0)
test_data_channel_10db_siso = np.concatenate((test_tdls_channel_10db_siso, test_cdls_channel_10db_siso), axis=0)
test_data_channel_10db_siso_labels = np.concatenate((test_label_tdl_channel_10db_siso, test_label_cdl_channel_10db_siso), axis=0)
test_data_channel_15db_siso = np.concatenate((test_tdls_channel_15db_siso, test_cdls_channel_15db_siso), axis=0)
test_data_channel_15db_siso_labels = np.concatenate((test_label_tdl_channel_15db_siso, test_label_cdl_channel_15db_siso), axis=0)
print("test_data_channel_0db_siso shape:", test_data_channel_0db_siso.shape)
print("test_data_channel_0db_siso_labels shape:", test_data_channel_0db_siso_labels.shape)
print("test_data_channel_5db_siso shape:", test_data_channel_5db_siso.shape)
print("test_data_channel_5db_siso_labels shape:", test_data_channel_5db_siso_labels.shape)
print("test_data_channel_10db_siso shape:", test_data_channel_10db_siso.shape)
print("test_data_channel_10db_siso_labels shape:", test_data_channel_10db_siso_labels.shape)
print("test_data_channel_15db_siso shape:", test_data_channel_15db_siso.shape)
print("test_data_channel_15db_siso_labels shape:", test_data_channel_15db_siso_labels.shape)
test_data_channel_0db_mimo = np.concatenate((test_tdls_channel_0db_mimo, test_cdls_channel_0db_mimo), axis=0)
test_data_channel_0db_mimo_labels = np.concatenate((test_label_tdl_channel_0db_mimo, test_label_cdl_channel_0db_mimo), axis=0)
test_data_channel_5db_mimo = np.concatenate((test_tdls_channel_5db_mimo, test_cdls_channel_5db_mimo), axis=0)
test_data_channel_5db_mimo_labels = np.concatenate((test_label_tdl_channel_5db_mimo, test_label_cdl_channel_5db_mimo), axis=0)
test_data_channel_10db_mimo = np.concatenate((test_tdls_channel_10db_mimo, test_cdls_channel_10db_mimo), axis=0)
test_data_channel_10db_mimo_labels = np.concatenate((test_label_tdl_channel_10db_mimo, test_label_cdl_channel_10db_mimo), axis=0)
test_data_channel_15db_mimo = np.concatenate((test_tdls_channel_15db_mimo, test_cdls_channel_15db_mimo), axis=0)
test_data_channel_15db_mimo_labels = np.concatenate((test_label_tdl_channel_15db_mimo, test_label_cdl_channel_15db_mimo), axis=0)
print("test_data_channel_0db_mimo shape:", test_data_channel_0db_mimo.shape)
print("test_data_channel_0db_mimo_labels shape:", test_data_channel_0db_mimo_labels.shape)
print("test_data_channel_5db_mimo shape:", test_data_channel_5db_mimo.shape)
print("test_data_channel_5db_mimo_labels shape:", test_data_channel_5db_mimo_labels.shape)
print("test_data_channel_10db_mimo shape:", test_data_channel_10db_mimo.shape)
print("test_data_channel_10db_mimo_labels shape:", test_data_channel_10db_mimo_labels.shape)
print("test_data_channel_15db_mimo shape:", test_data_channel_15db_mimo.shape)
print("test_data_channel_15db_mimo_labels shape:", test_data_channel_15db_mimo_labels.shape)

train_data_siso shape: (20000, 1, 1, 14, 192)
train_data_siso_labels shape: (20000,)
train_data_mimo shape: (20000, 1, 2, 14, 192)
train_data_mimo_labels shape: (20000,)
valid_data_mimo shape: (4000, 1, 2, 14, 192)
valid_data_mimo_labels shape: (4000,)
test_data_mimo shape: (10000, 1, 2, 14, 192)
test_data_mimo_labels shape: (10000,)
train_data_siso_channel shape: (20000, 1, 1, 1, 14, 192)
train_data_siso_channel_labels shape: (20000,)
test_data_siso_channel shape: (10000, 1, 1, 1, 14, 192)
test_data_siso_channel_labels shape: (10000,)
train_data_mimo_channel shape: (20000, 1, 1, 2, 14, 192)
train_data_mimo_channel_labels shape: (20000,)
test_data_mimo_channel shape: (10000, 1, 1, 2, 14, 192)
test_data_mimo_channel_labels shape: (10000,)
test_data_0db_siso shape: (10000, 1, 1, 14, 192)
test_data_0db_siso_labels shape: (10000,)
test_data_5db_siso shape: (10000, 1, 1, 14, 192)
test_data_5db_siso_labels shape: (10000,)
test_data_10db_siso shape: (10000, 1, 1, 14, 192)
test_data_10db_siso_

In [6]:

# train_data = train_data.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
# valid_data = valid_data.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
# test_data = test_data.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
# train_data = np.transpose(train_data, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
# valid_data = np.transpose(valid_data, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
# test_data = np.transpose(test_data, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)

# print(train_data.shape)  # Expected: (2000, 8, 14, 192)
# print(train_data_labels.shape)  # Expected: (2000,)
# print(test_data.shape)  # Expected: (8000, 8, 14, 192)
# print(test_data_labels.shape)  # Expected: (8000,)

# # check the variabes histogram (count the number of samples in each class)
# unique, counts = np.unique(train_data_labels, return_counts=True)
# print("Unique classes in train_data_labels:", unique)
# print("Counts of each class in train_data_labels:", counts)
# unique, counts = np.unique(test_data_labels, return_counts=True)
# print("Unique classes in test_data_labels:", unique)
# print("Counts of each class in test_data_labels:", counts)

# train_data_channel = train_data_channel.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
# valid_data_channel = valid_data_channel.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
# test_data_channel = test_data_channel.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
# train_data_channel = np.transpose(train_data_channel, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
# valid_data_channel = np.transpose(valid_data_channel, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
# test_data_channel = np.transpose(test_data_channel, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)

# print(train_data_channel.shape)  # Expected: (2000, 8, 14, 192)
# print(valid_data_channel.shape)  # Expected: (2000, 8, 14, 192)
# print(valid_data_channel_labels.shape)  # Expected: (2000,)
# print(test_data_channel.shape)  # Expected: (8000, 8, 14, 192)

train_data_siso = train_data_siso.reshape(-1, 1, 14, 192)  # Shape: (2000 * 8, 14, 192)
valid_data_siso = valid_data_siso.reshape(-1, 1, 14, 192)  # Shape: (2000 * 8, 14, 192)
test_data_siso = test_data_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
train_data_siso = np.transpose(train_data_siso, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
valid_data_siso = np.transpose(valid_data_siso, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
test_data_siso = np.transpose(test_data_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
print(train_data_siso.shape)  # Expected: (2000, 8, 14, 192)
print(train_data_siso_labels.shape)  # Expected: (2000,)
print(valid_data_siso.shape)  # Expected: (2000, 8, 14, 192)
print(valid_data_siso_labels.shape)  # Expected: (2000,)
print(test_data_siso.shape)  # Expected: (8000, 8, 14, 192)

train_data_siso_channel = train_data_siso_channel.reshape(-1, 1, 14, 192)  # Shape: (2000 * 8, 14, 192)
# valid_data_siso_channel = valid_data_siso_channel.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
test_data_siso_channel = test_data_siso_channel.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
train_data_siso_channel = np.transpose(train_data_siso_channel, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
# valid_data_siso_channel = np.transpose(valid_data_siso_channel, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
test_data_siso_channel = np.transpose(test_data_siso_channel, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
print(train_data_siso_channel.shape)  # Expected: (2000, 8, 14, 192)
print(train_data_siso_channel_labels.shape)  # Expected: (2000,)
# print(valid_data_siso_channel.shape)  # Expected: (2000, 8, 14, 192)
# print(valid_data_siso_channel_labels.shape)  # Expected: (2000,)
print(test_data_siso_channel.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_siso_channel_labels.shape)  # Expected: (8000,)

train_data_mimo_channel = train_data_mimo_channel.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
# valid_data_mimo_channel = valid_data_mimo_channel.reshape(-1, 4, 14, 192)  # Shape: (2000 * 8, 14, 192)
test_data_mimo_channel = test_data_mimo_channel.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
train_data_mimo_channel = np.transpose(train_data_mimo_channel, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
# valid_data_mimo_channel = np.transpose(valid_data_mimo_channel, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
test_data_mimo_channel = np.transpose(test_data_mimo_channel, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
print(train_data_mimo_channel.shape)  # Expected: (2000, 8, 14, 192)
print(train_data_mimo_channel_labels.shape)  # Expected: (2000,)
# print(valid_data_mimo_channel.shape)  # Expected: (2000, 8, 14, 192)
# print(valid_data_mimo_channel_labels.shape)  # Expected: (2000,)
print(test_data_mimo_channel.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_mimo_channel_labels.shape)  # Expected: (8000,)

test_data_0db_siso = test_data_0db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_5db_siso = test_data_5db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_10db_siso = test_data_10db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_15db_siso = test_data_15db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_0db_siso = np.transpose(test_data_0db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_5db_siso = np.transpose(test_data_5db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_10db_siso = np.transpose(test_data_10db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_15db_siso = np.transpose(test_data_15db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)

train_data_mimo = train_data_mimo.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
valid_data_mimo = valid_data_mimo.reshape(-1, 2, 14, 192)  # Shape: (2000 * 8, 14, 192)
test_data_mimo = test_data_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
train_data_mimo = np.transpose(train_data_mimo, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
valid_data_mimo = np.transpose(valid_data_mimo, (0, 3, 2, 1))  # Shape: (2000 * 8, 192, 14)
test_data_mimo = np.transpose(test_data_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
print(train_data_mimo.shape)  # Expected: (2000, 8, 14, 192)
print(train_data_mimo_labels.shape)  # Expected: (2000,)
print(valid_data_mimo.shape)  # Expected: (2000, 8, 14, 192)
print(valid_data_mimo_labels.shape)  # Expected: (2000,)
print(test_data_mimo.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_mimo_labels.shape)  # Expected: (8000,)

test_data_0db_mimo = test_data_0db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_5db_mimo = test_data_5db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_10db_mimo = test_data_10db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_15db_mimo = test_data_15db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_0db_mimo = np.transpose(test_data_0db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_5db_mimo = np.transpose(test_data_5db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_10db_mimo = np.transpose(test_data_10db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_15db_mimo = np.transpose(test_data_15db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)

test_data_channel_0db_siso = test_data_channel_0db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_5db_siso = test_data_channel_5db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_10db_siso = test_data_channel_10db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_15db_siso = test_data_channel_15db_siso.reshape(-1, 1, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_0db_siso = np.transpose(test_data_channel_0db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_5db_siso = np.transpose(test_data_channel_5db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_10db_siso = np.transpose(test_data_channel_10db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_15db_siso = np.transpose(test_data_channel_15db_siso, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_0db_mimo = test_data_channel_0db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_5db_mimo = test_data_channel_5db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_10db_mimo = test_data_channel_10db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_15db_mimo = test_data_channel_15db_mimo.reshape(-1, 2, 14, 192)  # Shape: (8000 * 8, 14, 192)
test_data_channel_0db_mimo = np.transpose(test_data_channel_0db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_5db_mimo = np.transpose(test_data_channel_5db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_10db_mimo = np.transpose(test_data_channel_10db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
test_data_channel_15db_mimo = np.transpose(test_data_channel_15db_mimo, (0, 3, 2, 1))  # Shape: (8000 * 8, 192, 14)
print(test_data_channel_0db_siso.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_0db_siso_labels.shape)  # Expected: (8000,)
print(test_data_channel_5db_siso.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_5db_siso_labels.shape)  # Expected: (8000,)
print(test_data_channel_10db_siso.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_10db_siso_labels.shape)  # Expected: (8000,)
print(test_data_channel_15db_siso.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_15db_siso_labels.shape)  # Expected: (8000,)
print(test_data_channel_0db_mimo.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_0db_mimo_labels.shape)  # Expected: (8000,)
print(test_data_channel_5db_mimo.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_5db_mimo_labels.shape)  # Expected: (8000,)
print(test_data_channel_10db_mimo.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_10db_mimo_labels.shape)  # Expected: (8000,)
print(test_data_channel_15db_mimo.shape)  # Expected: (8000, 8, 14, 192)
print(test_data_channel_15db_mimo_labels.shape)  # Expected: (8000,)

# # check the variabes histogram (count the number of samples in each class)
# unique, counts = np.unique(train_data_channel_labels, return_counts=True)
# print("Unique classes in train_data_labels_channel:", unique)
# print("Counts of each class in train_data_labels_channel:", counts)
# unique, counts = np.unique(valid_data_channel_labels, return_counts=True)
# print("Unique classes in valid_data_labels_channel:", unique)
# print("Counts of each class in valid_data_labels_channel:", counts)
# unique, counts = np.unique(test_data_channel_labels, return_counts=True)
# print("Unique classes in test_data_labels_channel:", unique)
# print("Counts of each class in test_data_labels_channel:", counts)
# check the variabes histogram (count the number of samples in each class)
unique, counts = np.unique(train_data_siso_labels, return_counts=True)
print("Unique classes in train_data_labels_siso:", unique)
print("Counts of each class in train_data_labels_siso:", counts)
unique, counts = np.unique(valid_data_siso_labels, return_counts=True)
print("Unique classes in valid_data_labels_siso:", unique)
print("Counts of each class in valid_data_labels_siso:", counts)
unique, counts = np.unique(test_data_siso_labels, return_counts=True)
print("Unique classes in test_data_labels_siso:", unique)
print("Counts of each class in test_data_labels_siso:", counts)
# check the variabes histogram (count the number of samples in each class)
unique, counts = np.unique(train_data_mimo_labels, return_counts=True)
print("Unique classes in train_data_labels_mimo:", unique)
print("Counts of each class in train_data_labels_mimo:", counts)
unique, counts = np.unique(valid_data_mimo_labels, return_counts=True)
print("Unique classes in valid_data_labels_mimo:", unique)
print("Counts of each class in valid_data_labels_mimo:", counts)
unique, counts = np.unique(test_data_mimo_labels, return_counts=True)
print("Unique classes in test_data_labels_mimo:", unique)
print("Counts of each class in test_data_labels_mimo:", counts)

unique, counts = np.unique(test_data_0db_siso_labels, return_counts=True)
print("Unique classes in test_data_0db_siso_labels:", unique)
print("Counts of each class in test_data_0db_siso_labels:", counts)

(20000, 192, 14, 1)
(20000,)
(4000, 192, 14, 1)
(4000,)
(10000, 192, 14, 1)
(20000, 192, 14, 1)
(20000,)
(10000, 192, 14, 1)
(10000,)
(20000, 192, 14, 2)
(20000,)
(10000, 192, 14, 2)
(10000,)
(20000, 192, 14, 2)
(20000,)
(4000, 192, 14, 2)
(4000,)
(10000, 192, 14, 2)
(10000,)
(10000, 192, 14, 1)
(10000,)
(10000, 192, 14, 1)
(10000,)
(10000, 192, 14, 1)
(10000,)
(10000, 192, 14, 1)
(10000,)
(10000, 192, 14, 2)
(10000,)
(10000, 192, 14, 2)
(10000,)
(10000, 192, 14, 2)
(10000,)
(10000, 192, 14, 2)
(10000,)
Unique classes in train_data_labels_siso: [0 1]
Counts of each class in train_data_labels_siso: [12000  8000]
Unique classes in valid_data_labels_siso: [0]
Counts of each class in valid_data_labels_siso: [4000]
Unique classes in test_data_labels_siso: [0 1]
Counts of each class in test_data_labels_siso: [6000 4000]
Unique classes in train_data_labels_mimo: [0 1]
Counts of each class in train_data_labels_mimo: [12000  8000]
Unique classes in valid_data_labels_mimo: [0]
Counts of each cla

In [7]:

# Define batch size (treating 8 as the batch size)
batch_size = 16

with tf.device('/CPU:0'):
    # # Create TensorFlow dataset
    # train_dataset = tf.data.Dataset.from_tensor_slices((train_data, train_data_labels))
    # train_dataset = train_dataset.shuffle(train_dataset.cardinality()).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # valid_dataset = tf.data.Dataset.from_tensor_slices((valid_data, valid_data_labels))
    # valid_dataset = valid_dataset.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    # test_dataset = tf.data.Dataset.from_tensor_slices((test_data, test_data_labels))
    # test_dataset = test_dataset.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # train_channel_dataset = tf.data.Dataset.from_tensor_slices((train_data_channel, train_data_channel_labels))
    # train_channel_dataset = train_channel_dataset.shuffle(train_channel_dataset.cardinality()).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    # valid_channel_dataset = tf.data.Dataset.from_tensor_slices((valid_data_channel, valid_data_channel_labels))
    # valid_channel_dataset = valid_channel_dataset.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    # test_channel_dataset = tf.data.Dataset.from_tensor_slices((test_data_channel, test_data_channel_labels))
    # test_channel_dataset = test_channel_dataset.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)


    train_dataset_siso = tf.data.Dataset.from_tensor_slices((train_data_siso, train_data_siso_labels))
    train_dataset_siso = train_dataset_siso.shuffle(train_dataset_siso.cardinality()).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    valid_dataset_siso = tf.data.Dataset.from_tensor_slices((valid_data_siso, valid_data_siso_labels))
    valid_dataset_siso = valid_dataset_siso.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_siso = tf.data.Dataset.from_tensor_slices((test_data_siso, test_data_siso_labels))
    test_dataset_siso = test_dataset_siso.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    train_dataset_mimo = tf.data.Dataset.from_tensor_slices((train_data_mimo, train_data_mimo_labels))
    train_dataset_mimo = train_dataset_mimo.shuffle(train_dataset_mimo.cardinality()).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    valid_dataset_mimo = tf.data.Dataset.from_tensor_slices((valid_data_mimo, valid_data_mimo_labels))
    valid_dataset_mimo = valid_dataset_mimo.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_mimo = tf.data.Dataset.from_tensor_slices((test_data_mimo, test_data_mimo_labels))
    test_dataset_mimo = test_dataset_mimo.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    train_dataset_mimo_channel = tf.data.Dataset.from_tensor_slices((train_data_mimo_channel, train_data_mimo_channel_labels))
    train_dataset_mimo_channel = train_dataset_mimo_channel.shuffle(train_dataset_mimo_channel.cardinality()).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # valid_dataset_mimo_channel = tf.data.Dataset.from_tensor_slices((valid_data_mimo_channel, valid_data_mimo_channel_labels))
    # valid_dataset_mimo_channel = valid_dataset_mimo_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_mimo_channel = tf.data.Dataset.from_tensor_slices((test_data_mimo_channel, test_data_mimo_channel_labels))
    test_dataset_mimo_channel = test_dataset_mimo_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    train_dataset_siso_channel = tf.data.Dataset.from_tensor_slices((train_data_siso_channel, train_data_siso_channel_labels))
    train_dataset_siso_channel = train_dataset_siso_channel.shuffle(train_dataset_siso_channel.cardinality()).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # valid_dataset_siso_channel = tf.data.Dataset.from_tensor_slices((valid_data_siso_channel, valid_data_siso_channel_labels))
    # valid_dataset_siso_channel = valid_dataset_siso_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_siso_channel = tf.data.Dataset.from_tensor_slices((test_data_siso_channel, test_data_siso_channel_labels))
    test_dataset_siso_channel = test_dataset_siso_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    
    test_dataset_0db_siso = tf.data.Dataset.from_tensor_slices((test_data_0db_siso, test_data_0db_siso_labels))
    test_dataset_0db_siso = test_dataset_0db_siso.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_5db_siso = tf.data.Dataset.from_tensor_slices((test_data_5db_siso, test_data_5db_siso_labels))
    test_dataset_5db_siso = test_dataset_5db_siso.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_10db_siso = tf.data.Dataset.from_tensor_slices((test_data_10db_siso, test_data_10db_siso_labels))
    test_dataset_10db_siso = test_dataset_10db_siso.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_15db_siso = tf.data.Dataset.from_tensor_slices((test_data_15db_siso, test_data_15db_siso_labels))
    test_dataset_15db_siso = test_dataset_15db_siso.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_0db_mimo = tf.data.Dataset.from_tensor_slices((test_data_0db_mimo, test_data_0db_mimo_labels))
    test_dataset_0db_mimo = test_dataset_0db_mimo.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_5db_mimo = tf.data.Dataset.from_tensor_slices((test_data_5db_mimo, test_data_5db_mimo_labels))
    test_dataset_5db_mimo = test_dataset_5db_mimo.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_10db_mimo = tf.data.Dataset.from_tensor_slices((test_data_10db_mimo, test_data_10db_mimo_labels))
    test_dataset_10db_mimo = test_dataset_10db_mimo.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_15db_mimo = tf.data.Dataset.from_tensor_slices((test_data_15db_mimo, test_data_15db_mimo_labels))
    test_dataset_15db_mimo = test_dataset_15db_mimo.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_0db_mimo_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_0db_mimo, test_data_channel_0db_mimo_labels))
    test_dataset_0db_mimo_channel = test_dataset_0db_mimo_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_5db_mimo_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_5db_mimo, test_data_channel_5db_mimo_labels))
    test_dataset_5db_mimo_channel = test_dataset_5db_mimo_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_10db_mimo_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_10db_mimo, test_data_channel_10db_mimo_labels))
    test_dataset_10db_mimo_channel = test_dataset_10db_mimo_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_15db_mimo_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_15db_mimo, test_data_channel_15db_mimo_labels))
    test_dataset_15db_mimo_channel = test_dataset_15db_mimo_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_0db_siso_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_0db_siso, test_data_channel_0db_siso_labels))
    test_dataset_0db_siso_channel = test_dataset_0db_siso_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_5db_siso_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_5db_siso, test_data_channel_5db_siso_labels))
    test_dataset_5db_siso_channel = test_dataset_5db_siso_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_10db_siso_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_10db_siso, test_data_channel_10db_siso_labels))
    test_dataset_10db_siso_channel = test_dataset_10db_siso_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_dataset_15db_siso_channel = tf.data.Dataset.from_tensor_slices((test_data_channel_15db_siso, test_data_channel_15db_siso_labels))
    test_dataset_15db_siso_channel = test_dataset_15db_siso_channel.cache().batch(batch_size).prefetch(tf.data.AUTOTUNE)



In [8]:
# import cvnn.layers as complex_layers
# import tensorflow as tf
# batch_size = 16  # Define batch size
# def VGGComplexNet(input_shape=(192, 14, 4),
#            output_channels=4,
#            dim=4,
#            n_downsamplings=1,
#            n_blocks=1,
#            num_classes=2,
#            num_rx_tx=4,
#            batch_size=1,
#            norm='instance_norm'):
#     inputs = complex_layers.complex_input(shape=input_shape, batch_size=batch_size)
#     h = complex_layers.ComplexConv2D(dim, kernel_size=1, padding='same', activation='cart_relu')(inputs)
#     h = complex_layers.ComplexConv2D(dim, kernel_size=1, padding='same', activation='cart_relu')(h)
#     h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(h)

#     for i in range(n_downsamplings):
#         h = complex_layers.ComplexConv2D(dim * (2 ** (i+1)), kernel_size=1, padding='same', activation='cart_relu')(h)
#         h = complex_layers.ComplexConv2D(dim * (2 ** (i+1)), kernel_size=1, padding='same', activation='cart_relu')(h)
#         h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(h)

#     for _ in range(n_blocks):
#         res = h
#         h = complex_layers.ComplexConv2D(h.shape[-1], kernel_size=1, padding='same', activation='cart_relu')(h)
#         h = tf.keras.layers.Add()([h, res])

#     h = complex_layers.ComplexFlatten()(h)
#     h = complex_layers.ComplexDense(512, activation='cart_relu')(h)
#     h = complex_layers.ComplexDense(256, activation='cart_relu')(h)
#     h = complex_layers.ComplexDense(num_classes, activation='convert_to_real_with_abs')(h)

#     model = tf.keras.Model(inputs=inputs, outputs=h)
#     return model

# # Create the model
# model = VGGComplexNet(input_shape=(192, 14, 4), output_channels=4, dim=4, n_downsamplings=1, n_blocks=1, num_classes=2, num_rx_tx=4, batch_size=batch_size)
# model.summary()


import cvnn.layers as complex_layers
import tensorflow as tf

batch_size = 16  # Define batch size

def VGGComplexNet(input_shape=(192, 14, 4),
           output_channels=4,
           dim=4,
           n_downsamplings=1,
           n_blocks=1,
           num_classes=2,
           num_rx_tx=4,
           batch_size=1,
           norm='instance_norm'):
    inputs = complex_layers.complex_input(shape=input_shape, batch_size=batch_size)
    h = complex_layers.ComplexConv2D(dim, kernel_size=1, padding='same', activation='cart_relu')(inputs)
    h = complex_layers.ComplexConv2D(dim, kernel_size=1, padding='same', activation='cart_relu')(h)
    # h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(h)

    for i in range(n_downsamplings):
        dim *= 2
        print(f"dim: {dim}, h.shape[2]: {h.shape[2]}")
        if h.shape[2] % 2 == 0:
            h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(h)
        else:
            h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 1), padding='same')(h)
        for j in range(2):  # or whatever number of convs you want per block
            h = complex_layers.ComplexConv2D(dim, kernel_size=1, padding='same', activation='cart_relu')(h)

    for _ in range(n_blocks):
        res = h
        h = complex_layers.ComplexConv2D(h.shape[-1], kernel_size=1, padding='same', activation='cart_relu')(h)
        h = tf.keras.layers.Add()([h, res])

    h = complex_layers.ComplexFlatten()(h)
    h = complex_layers.ComplexDense(512, activation='cart_relu')(h)
    h = complex_layers.ComplexDense(256, activation='cart_relu')(h)
    h = complex_layers.ComplexDense(num_classes, activation='convert_to_real_with_abs')(h)

    model = tf.keras.Model(inputs=inputs, outputs=h)
    return model

# Create the model
model = VGGComplexNet(input_shape=(192, 14, 4), output_channels=4, dim=4, n_downsamplings=1, n_blocks=1, num_classes=2, num_rx_tx=4, batch_size=batch_size)
model.summary()
model.save('vgg_complex_net.keras')

n_downsamplings = 3
n_blocks = 0
num_rx_tx = 4  # Number of receive and transmit antennas
# Explicitly set the batch size to 8
input_shape = (192, 14, 2)  # (4, 192, 14)
num_classes = 2
dim = 4
model = VGGComplexNet(
    batch_size=batch_size,
    input_shape=input_shape,
    num_rx_tx=num_rx_tx,
    num_classes=num_classes,
    dim=dim,
    n_downsamplings=n_downsamplings,
    n_blocks=n_blocks,
    norm='instance_norm'
)
# model.compile(optimizer='adam', metrics=['accuracy'],
#               loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
model.summary()
# save the model
# model.save('VGGNet.keras')


def SimpleComplexNet(input_shape=(192, 14, 2), num_classes=2, batch_size=16, activation='crelu'):
    inputs = complex_layers.complex_input(shape=input_shape, batch_size=batch_size)
    x = complex_layers.ComplexConv2D(8, kernel_size=1, padding='same', activation=activation)(inputs)
    x = complex_layers.ComplexFlatten()(x)
    x = complex_layers.ComplexDense(num_classes, activation='convert_to_real_with_abs')(x)
    model = tf.keras.Model(inputs=inputs, outputs=x)
    return model

# # Example usage:
# batch_size = 16
# input_shape = (192, 14, 2)
# num_classes = 2
# model = SimpleComplexNet(input_shape=input_shape, num_classes=num_classes, batch_size=batch_size)
# model.summary()


dim: 8, h.shape[2]: 14
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (ComplexInput)         [(16, 192, 14, 4)]   0           []                               
                                                                                                  
 complex_conv2d (ComplexConv2D)  (16, 192, 14, 4)    40          ['input_1[0][0]']                
                                                                                                  
 complex_conv2d_1 (ComplexConv2  (16, 192, 14, 4)    40          ['complex_conv2d[0][0]']         
 D)                                                                                               
                                                                                                  
 complex_max_pooling2d (Complex  (16, 96, 7, 4)      0           ['comp

In [13]:
import cvnn.layers as complex_layers
import tensorflow as tf
import os
from datetime import datetime
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from cvnn.losses import ComplexAverageCrossEntropy
from cvnn.metrics import ComplexAccuracy
class ComplexSparseCategoricalAccuracy(SparseCategoricalAccuracy):
    def __init__(self, name='complex_sparse_categorical_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.convert_to_tensor(y_pred)
        y_true = tf.convert_to_tensor(y_true)
        if y_pred.dtype.is_complex:
            y_pred = (tf.math.real(y_pred) + tf.math.imag(y_pred)) / 2
        if y_true.dtype.is_complex:
            y_true = tf.math.real(y_true)
        super().update_state(y_true, y_pred, sample_weight=sample_weight)
from tensorflow.keras.losses import sparse_categorical_crossentropy

class ComplexAverageSparseCategoricalCrossentropy(SparseCategoricalCrossentropy):
    # def call(self, y_true, y_pred):
    #     # If y_pred is complex, convert to real
    #     if y_pred.dtype.is_complex:
    #         y_pred = (tf.math.real(y_pred) + tf.math.imag(y_pred)) / 2
    #     return super().call(y_true, y_pred)
    def call(self, y_true, y_pred):
        real_loss = sparse_categorical_crossentropy(y_true, tf.math.real(y_pred), from_logits=True)
        if y_pred.dtype.is_complex:
            imag_loss = sparse_categorical_crossentropy(y_true, tf.math.imag(y_pred), from_logits=True)
        else:
            imag_loss = real_loss
        return (real_loss + imag_loss) / 2.
batch_size = 16
activations = ['modrelu', 'crelu', 'zrelu', 'cart_relu']
# activations = ['crelu', 'zrelu', 'cart_relu']
# activations = ['crelu']
# activations = ['cart_relu']
# activations = ['modrelu']
# activations = ['zrelu']

for activation in activations:
    print(f"\n=== Training with activation: {activation} ===\n")
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(f"cvnn_logs/mimo/{activation}", timestamp)
    save_dir = os.path.join(f"cvnn_models/mimo/{activation}", timestamp)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(save_dir, exist_ok=True)

    def VGGComplexNet(input_shape=(192, 14, 2),
               output_channels=4,
               dim=16,
               n_downsamplings=3,
               n_blocks=0,
               num_classes=2,
               num_rx_tx=4,
               batch_size=16,
               norm='instance_norm'):
        inputs = complex_layers.complex_input(shape=input_shape, batch_size=batch_size)
        h = complex_layers.ComplexConv2D(dim, kernel_size=(1,1), strides=(1,1), dilation_rate=1, padding='same', activation=activation)(inputs)
        for i in range(n_downsamplings):
            dim *= 2
            if h.shape[2] % 2 == 0:
                h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(h)
            else:
                h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 1), padding='same')(h)
            value = 2 if i <= 0 else 3
            for j in range(min(value, 3)):
                h = complex_layers.ComplexConv2D(dim, kernel_size=(1,1), strides=(1,1), dilation_rate=1, padding='same', activation=activation)(h)
        h = complex_layers.ComplexFlatten()(h)
        h = complex_layers.ComplexDense(512, activation=activation)(h)
        h = complex_layers.ComplexDense(256, activation=activation)(h)
        h = complex_layers.ComplexDense(num_classes, activation='convert_to_real_with_abs')(h)
        model = tf.keras.Model(inputs=inputs, outputs=h)
        return model

    # Model and datasets
    # model = VGGComplexNet()
    
    model.summary()

    writer = tf.summary.create_file_writer(log_dir)
    # loss_fn = ComplexAverageCrossEntropy()
    loss_fn = ComplexAverageSparseCategoricalCrossentropy()
    optimizer = tf.keras.optimizers.SGD(learning_rate=0.0005, momentum=0.9)
    train_accuracy = ComplexSparseCategoricalAccuracy()
    val_accuracy = ComplexSparseCategoricalAccuracy()
    test_accuracy = ComplexSparseCategoricalAccuracy()
    # train_accuracy = ComplexAccuracy()
    # val_accuracy = ComplexAccuracy()
    # test_accuracy = ComplexAccuracy()
    # Use your datasets here
    train_dataset = train_dataset_mimo
    valid_dataset = valid_dataset_mimo
    test_dataset = test_dataset_mimo
    epochs = 100
    best_val_loss = float('inf')
    best_test_loss = float('inf')
    patience = 5  # Number of epochs to wait for improvement
    wait = 0       # Counter for early stopping

    @tf.function
    def train_step(x_batch, y_batch):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_accuracy.update_state(y_true=y_batch, y_pred=logits)
        return loss

    @tf.function
    def val_step(x_batch, y_batch):
        logits = model(x_batch, training=False)
        loss = loss_fn(y_batch, logits)
        val_accuracy.update_state(y_true=y_batch, y_pred=logits)
        return loss

    @tf.function
    def test_step(x_batch, y_batch):
        logits = model(x_batch, training=False)
        loss = loss_fn(y_batch, logits)
        test_accuracy.update_state(y_true=y_batch, y_pred=logits)
        return loss

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        epoch_loss = 0.0
        train_accuracy.reset_state()

        for x_batch, y_batch in train_dataset:
            loss = train_step(x_batch, y_batch)
            epoch_loss += loss.numpy()
        epoch_loss /= len(train_dataset)
        train_acc = train_accuracy.result().numpy()

        with writer.as_default():
            tf.summary.scalar("training loss", epoch_loss, step=epoch)
            tf.summary.scalar("training accuracy", train_acc, step=epoch)
        
        val_loss = 0.0
        val_accuracy.reset_state()
        for x_val, y_val in valid_dataset:
            loss = val_step(x_val, y_val)
            val_loss += loss.numpy()
        val_loss /= len(valid_dataset)
        val_acc = val_accuracy.result().numpy()
        with writer.as_default():
            tf.summary.scalar("validation loss", val_loss, step=epoch)
            tf.summary.scalar("validation accuracy", val_acc, step=epoch)

        test_loss = 0.0
        test_accuracy.reset_state()
        for x_test, y_test in test_dataset:
            loss = test_step(x_test, y_test)
            test_loss += loss.numpy()
        test_loss /= len(test_dataset)
        test_acc = test_accuracy.result().numpy()
        with writer.as_default():
            tf.summary.scalar("test loss", test_loss, step=epoch)
            tf.summary.scalar("test accuracy", test_acc, step=epoch)

        keras_model_path = os.path.join(save_dir, f"model_epoch_{epoch + 1:02d}.keras")
        model.save(keras_model_path)
        print(f"Model for epoch {epoch + 1} saved at {keras_model_path}")

        print(f"Training Loss: {epoch_loss:.4f}, Training Accuracy: {train_acc:.4f}")
        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")
        print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

        if val_loss < best_val_loss or epoch == 0:
            best_val_loss = val_loss
            best_keras_model_path = os.path.join(save_dir, "best_model.keras")
            model.save(best_keras_model_path)
            print("Best model saved in Keras format based on validation loss")
        #     wait = 0  # Reset patience counter
        # else:
        #     wait += 1
        #     if wait >= patience:
        #         print(f"Early stopping triggered at epoch {epoch + 1}")
        #         break

        if test_loss < best_test_loss or epoch == 0:
            best_test_loss = test_loss
            best_keras_model_path = os.path.join(save_dir, "best_model_test.keras")
            model.save(best_keras_model_path)
            print("Best model saved in Keras format based on test loss")
            wait = 0  # Reset patience counter
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break


=== Training with activation: modrelu ===

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (ComplexInput)      [(16, 192, 14, 2)]        0         
                                                                 
 complex_conv2d_5 (ComplexCo  (16, 192, 14, 4)         24        
 nv2D)                                                           
                                                                 
 complex_conv2d_6 (ComplexCo  (16, 192, 14, 4)         40        
 nv2D)                                                           
                                                                 
 complex_max_pooling2d_1 (Co  (16, 96, 7, 4)           0         
 mplexMaxPooling2D)                                              
                                                                 
 complex_conv2d_7 (ComplexCo  (16, 96, 7, 8)           80        
 nv2D)         

In [14]:
import os
import numpy as np
import tensorflow as tf
from datetime import datetime
import cvnn.layers as complex_layers
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.metrics import SparseCategoricalAccuracy
from cvnn.losses import ComplexAverageCrossEntropy
from cvnn.metrics import ComplexAccuracy
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc

# --- Custom Metrics and Losses ---
class ComplexSparseCategoricalAccuracy(SparseCategoricalAccuracy):
    def __init__(self, name='complex_sparse_categorical_accuracy', **kwargs):
        super().__init__(name=name, **kwargs)

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.convert_to_tensor(y_pred)
        y_true = tf.convert_to_tensor(y_true)
        if y_pred.dtype.is_complex:
            y_pred = (tf.math.real(y_pred) + tf.math.imag(y_pred)) / 2
        if y_true.dtype.is_complex:
            y_true = tf.math.real(y_true)
        super().update_state(y_true, y_pred, sample_weight=sample_weight)
from tensorflow.keras.losses import sparse_categorical_crossentropy

class ComplexAverageSparseCategoricalCrossentropy(SparseCategoricalCrossentropy):
    # def call(self, y_true, y_pred):
    #     # If y_pred is complex, convert to real
    #     if y_pred.dtype.is_complex:
    #         y_pred = (tf.math.real(y_pred) + tf.math.imag(y_pred)) / 2
    #     return super().call(y_true, y_pred)
    def call(self, y_true, y_pred):
        real_loss = sparse_categorical_crossentropy(y_true, tf.math.real(y_pred), from_logits=True)
        if y_pred.dtype.is_complex:
            imag_loss = sparse_categorical_crossentropy(y_true, tf.math.imag(y_pred), from_logits=True)
        else:
            imag_loss = real_loss
        return (real_loss + imag_loss) / 2.

# --- ResNet50 Block Definitions ---
def compact_complex_identity_block(input_tensor, filters, kernel_size=1, activation='crelu'):
    f1 = filters
    x = complex_layers.ComplexConv2D(f1, (kernel_size, kernel_size), padding='same', activation=activation)(input_tensor)
    x = complex_layers.ComplexConv2D(f1, (kernel_size, kernel_size), padding='same', activation=None)(x)
    x = tf.keras.layers.Add()([x, input_tensor])
    x = tf.keras.layers.Activation('linear')(x)
    return x

def compact_complex_conv_block(input_tensor, filters, kernel_size=1, strides=(1, 1), activation='crelu'):
    f1 = filters
    x = complex_layers.ComplexConv2D(f1, (kernel_size, kernel_size), strides=strides, padding='same', activation=activation)(input_tensor)
    x = complex_layers.ComplexConv2D(f1, (kernel_size, kernel_size), padding='same', activation=None)(x)
    shortcut = complex_layers.ComplexConv2D(f1, (1, 1), strides=strides, padding='same', activation=None)(input_tensor)
    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.Activation('linear')(x)
    return x

def CompactComplexResNet(input_shape=(192, 14, 2), num_classes=2, batch_size=16, activation='crelu'):
    inputs = complex_layers.complex_input(shape=input_shape, batch_size=batch_size)
    x = complex_layers.ComplexConv2D(16, (3, 3), strides=(1, 1), padding='same', activation=activation)(inputs)
    # Stage 1
    x = compact_complex_conv_block(x, 16, strides=(1, 1), activation=activation)
    x = compact_complex_identity_block(x, 16, activation=activation)
    # Stage 2
    if x.shape[2] % 2 == 0:
        x = complex_layers.ComplexMaxPooling2D((2, 2), strides=(2, 2), padding='same')(x)
    else:
        x = complex_layers.ComplexMaxPooling2D((2, 2), strides=(2, 1), padding='same')(x)
    x = compact_complex_conv_block(x, 32, strides=(1, 1), activation=activation)
    x = compact_complex_identity_block(x, 32, activation=activation)
    # Stage 3
    if x.shape[2] % 2 == 0:
        x = complex_layers.ComplexMaxPooling2D((2, 2), strides=(2, 2), padding='same')(x)
    else:
        x = complex_layers.ComplexMaxPooling2D((2, 2), strides=(2, 1), padding='same')(x)
    x = compact_complex_conv_block(x, 64, strides=(1, 1), activation=activation)
    x = compact_complex_identity_block(x, 64, activation=activation)
    # Stage 4
    if x.shape[2] % 2 == 0:
        x = complex_layers.ComplexMaxPooling2D((2, 2), strides=(2, 2), padding='same')(x)
    else:
        x = complex_layers.ComplexMaxPooling2D((2, 2), strides=(2, 1), padding='same')(x)
    x = compact_complex_conv_block(x, 128, strides=(1, 1), activation=activation)
    x = compact_complex_identity_block(x, 128, activation=activation)
    # Head
    x = complex_layers.ComplexFlatten()(x)
    x = complex_layers.ComplexDense(128, activation=activation)(x)  # reduced from 512 to 128
    x = complex_layers.ComplexDense(num_classes, activation='convert_to_real_with_abs')(x)
    model = tf.keras.Model(inputs=inputs, outputs=x)
    return model

# --- Training and Evaluation ---
def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5,5))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(np.unique(y_true)))
    plt.xticks(tick_marks, tick_marks)
    plt.yticks(tick_marks, tick_marks)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.show()

def plot_roc_curve(y_true, y_prob, title="ROC Curve"):
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(title)
        plt.legend(loc="lower right")
        plt.show()
    else:
        for i in range(y_prob.shape[1]):
            fpr, tpr, _ = roc_curve((y_true == i).astype(int), y_prob[:, i])
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, lw=2, label=f'Class {i} (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(title)
        plt.legend(loc="lower right")
        plt.show()

# --- Example Usage ---
batch_size = 16
input_shape = (192, 14, 2)
num_classes = 2
activation = 'crelu'
epochs = 100

# Use your datasets here
# train_dataset = train_dataset_mimo_channel
# valid_dataset = valid_dataset_mimo_channel
# test_dataset = test_dataset_mimo_channel
train_dataset = train_dataset_mimo
valid_dataset = valid_dataset_mimo
test_dataset = test_dataset_mimo
model = CompactComplexResNet(input_shape=input_shape, num_classes=num_classes, batch_size=batch_size, activation=activation)
model.summary()
model.save('CompactComplexResNet.keras')
loss_fn = ComplexAverageSparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1)
train_accuracy = ComplexSparseCategoricalAccuracy()
val_accuracy = ComplexSparseCategoricalAccuracy()
test_accuracy = ComplexSparseCategoricalAccuracy()

log_dir = os.path.join(f"logs_{activation}", datetime.now().strftime("%Y%m%d-%H%M%S"))
save_dir = os.path.join(f"saved_models_{activation}", datetime.now().strftime("%Y%m%d-%H%M%S"))
os.makedirs(log_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)
writer = tf.summary.create_file_writer(log_dir)

@tf.function
def train_step(x_batch, y_batch):
    with tf.GradientTape() as tape:
        logits = model(x_batch, training=True)
        loss = loss_fn(y_batch, logits)
        tf.print("Loss:", loss)
        tf.print("Logits:", logits)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    train_accuracy.update_state(y_true=y_batch, y_pred=logits)
    return loss

@tf.function
def val_step(x_batch, y_batch):
    logits = model(x_batch, training=False)
    loss = loss_fn(y_batch, logits)
    val_accuracy.update_state(y_true=y_batch, y_pred=logits)
    return loss

@tf.function
def test_step(x_batch, y_batch):
    logits = model(x_batch, training=False)
    loss = loss_fn(y_batch, logits)
    test_accuracy.update_state(y_true=y_batch, y_pred=logits)
    return loss

best_val_loss = float('inf')
for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    epoch_loss = 0.0
    train_accuracy.reset_state()
    for x_batch, y_batch in train_dataset:
        loss = train_step(x_batch, y_batch)
        epoch_loss += loss.numpy()
    epoch_loss /= len(train_dataset)
    train_acc = train_accuracy.result().numpy()
    with writer.as_default():
        tf.summary.scalar("training loss", epoch_loss, step=epoch)
        tf.summary.scalar("training accuracy", train_acc, step=epoch)

    val_loss = 0.0
    val_accuracy.reset_state()
    for x_val, y_val in valid_dataset:
        loss = val_step(x_val, y_val)
        val_loss += loss.numpy()
    val_loss /= len(valid_dataset)
    val_acc = val_accuracy.result().numpy()
    with writer.as_default():
        tf.summary.scalar("validation loss", val_loss, step=epoch)
        tf.summary.scalar("validation accuracy", val_acc, step=epoch)

    test_loss = 0.0
    test_accuracy.reset_state()
    for x_test, y_test in test_dataset:
        loss = test_step(x_test, y_test)
        test_loss += loss.numpy()
    test_loss /= len(test_dataset)
    test_acc = test_accuracy.result().numpy()
    with writer.as_default():
        tf.summary.scalar("test loss", test_loss, step=epoch)
        tf.summary.scalar("test accuracy", test_acc, step=epoch)

    keras_model_path = os.path.join(save_dir, f"model_epoch_{epoch + 1:02d}.keras")
    model.save(keras_model_path)
    print(f"Model for epoch {epoch + 1} saved at {keras_model_path}")

    print(f"Training Loss: {epoch_loss:.4f}, Training Accuracy: {train_acc:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

    if val_loss < best_val_loss or epoch == 0:
        best_val_loss = val_loss
        best_keras_model_path = os.path.join(save_dir, "best_model.keras")
        model.save(best_keras_model_path)
        print("Best model saved in Keras format based on validation loss")

# --- Evaluation and Visualization ---
best_model_path = os.path.join(save_dir, "best_model.keras")
model = tf.keras.models.load_model(best_model_path, compile=False)

y_true = []
y_pred = []
y_prob = []
for x_batch, y_batch in test_dataset:
    logits = model(x_batch, training=False)
    if num_classes > 2:
        probs = tf.nn.softmax(logits).numpy()
        preds = np.argmax(probs, axis=1)
    else:
        probs = tf.nn.sigmoid(logits).numpy().flatten()
        preds = (probs > 0.5).astype(int)
    y_true.extend(y_batch.numpy())
    y_pred.extend(preds)
    y_prob.extend(probs)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

plot_confusion_matrix(y_true, y_pred)
plot_roc_curve(y_true, y_prob)

Model: "model_2"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (ComplexInput)         [(16, 192, 14, 2)]   0           []                               
                                                                                                  
 complex_conv2d_13 (ComplexConv  (16, 192, 14, 16)   608         ['input_3[0][0]']                
 2D)                                                                                              
                                                                                                  
 complex_conv2d_14 (ComplexConv  (16, 192, 14, 16)   544         ['complex_conv2d_13[0][0]']      
 2D)                                                                                              
                                                                                            

KeyboardInterrupt: 

In [ ]:

# # Define directories for saving logs and models
# nlos_los_log_dir = os.path.join("logs_nlos_los_complex", datetime.now().strftime("%Y%m%d-%H%M%S"))
# nlos_los_save_dir = os.path.join("saved_models_nlos_los_complex", datetime.now().strftime("%Y%m%d-%H%M%S"))
# channel_log_dir = os.path.join("logs_channel_complex", datetime.now().strftime("%Y%m%d-%H%M%S"))
# channel_save_dir = os.path.join("saved_models_channel_complex", datetime.now().strftime("%Y%m%d-%H%M%S"))
# # Create directories if they don't exist
# os.makedirs(nlos_los_log_dir, exist_ok=True)
# os.makedirs(channel_log_dir, exist_ok=True)
# os.makedirs(channel_save_dir, exist_ok=True)
# os.makedirs(nlos_los_save_dir, exist_ok=True)


In [ ]:
# # TensorBoard writer
# writer = tf.summary.create_file_writer(nlos_los_log_dir)

# # Define the loss function
# # loss_fn = tf.keras.losses.BinaryCrossentropy() if num_classes == 2 else tf.keras.losses.SparseCategoricalCrossentropy()
# loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
# # Define the optimizer
# # optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002)
# optimizer = tf.keras.optimizers.SGD(learning_rate=0.0005, momentum=0.9)  # SGD with momentum

# # Define metrics
# # train_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# # val_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# # test_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# train_accuracy = SparseCategoricalAccuracy()
# val_accuracy = SparseCategoricalAccuracy()
# test_accuracy = SparseCategoricalAccuracy()

# # Optimize dataset loading
# # Define training and validation steps with @tf.function for graph execution
# @tf.function
# def train_step(x_batch, y_batch):
#     with tf.GradientTape() as tape:
#         logits = model(x_batch, training=True)
#         loss = loss_fn(y_batch, logits)
#     gradients = tape.gradient(loss, model.trainable_variables)
#     optimizer.apply_gradients(zip(gradients, model.trainable_variables))
#     train_accuracy.update_state(y_batch, logits)
#     return loss

# @tf.function
# def val_step(x_batch, y_batch):
#     logits = model(x_batch, training=False)
#     loss = loss_fn(y_batch, logits)
#     val_accuracy.update_state(y_batch, logits)
#     return loss

# @tf.function
# def test_step(x_batch, y_batch):
#     logits = model(x_batch, training=False)
#     loss = loss_fn(y_batch, logits)
#     test_accuracy.update_state(y_batch, logits)
#     return loss

In [ ]:
# train_dataset = train_dataset_mimo
# valid_dataset = valid_dataset_mimo
# test_dataset = test_dataset_mimo

# epochs = 100  # Set the number of epochs
# best_val_loss = float('inf')  # Track the best validation loss
# best_test_loss = float('inf')  # Track the best test loss
# patience = 10  # Early stopping patience

# for epoch in range(epochs):
#     print(f"Epoch {epoch + 1}/{epochs}")
#     epoch_loss = 0.0
#     train_accuracy.reset_state()

#     # Training loop
#     for x_batch, y_batch in train_dataset:
#         loss = train_step(x_batch, y_batch)
#         epoch_loss += loss.numpy()
#     epoch_loss /= len(train_dataset)
#     train_acc = train_accuracy.result().numpy()

#     # Log training metrics to TensorBoard
#     with writer.as_default():
#         tf.summary.scalar("training loss", epoch_loss, step=epoch)
#         tf.summary.scalar("training accuracy", train_acc, step=epoch)

#     # Validation loop
#     val_loss = 0.0
#     val_accuracy.reset_state()
#     for x_val, y_val in valid_dataset:
#         loss = val_step(x_val, y_val)
#         val_loss += loss.numpy()
#     val_loss /= len(valid_dataset)
#     val_acc = val_accuracy.result().numpy()
#     with writer.as_default():
#         tf.summary.scalar("validation loss", val_loss, step=epoch)
#         tf.summary.scalar("validation accuracy", val_acc, step=epoch)

#     # Test loop
#     test_loss = 0.0
#     test_accuracy.reset_state()
#     for x_test, y_test in test_dataset:
#         loss = test_step(x_test, y_test)
#         test_loss += loss.numpy()
#     test_loss /= len(test_dataset)
#     test_acc = test_accuracy.result().numpy()
#     with writer.as_default():
#         tf.summary.scalar("test loss", test_loss, step=epoch)
#         tf.summary.scalar("test accuracy", test_acc, step=epoch)

#     # Save the Keras model at each epoch
#     keras_model_path = os.path.join(nlos_los_save_dir, f"model_epoch_{epoch + 1:02d}.keras")
#     model.save(keras_model_path)
#     print(f"Model for epoch {epoch + 1} saved at {keras_model_path}")

#     print(f"Training Loss: {epoch_loss:.4f}, Training Accuracy: {train_acc:.4f}")
#     print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")
#     print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

#     # Save the best model based on validation loss
#     if val_loss < best_val_loss or epoch == 0:
#         best_val_loss = val_loss
#         best_keras_model_path = os.path.join(nlos_los_save_dir, "best_model.keras")
#         model.save(best_keras_model_path)
#         print("Best model saved in Keras format based on validation loss")

#     # Save the best model based on test loss
#     if test_loss < best_test_loss or epoch == 0:
#         best_test_loss = test_loss
#         best_keras_model_path = os.path.join(nlos_los_save_dir, "best_model_test.keras")
#         model.save(best_keras_model_path)
#         print("Best model saved in Keras format based on test loss")
#         # patience = 10
#     # else:
#     #     patience -= 1
#     #     if patience == 0:
#     #         print("Early stopping triggered")
#     #         break

In [ ]:
# import onnxruntime as ort  # ONNX Runtime for inference

# # Load the best model
# best_model_path = os.path.join(nlos_los_save_dir, "best_model.onnx")
# if os.path.exists(best_model_path):
#     print(f"Loading best model from {best_model_path}")
#     # Use ONNX Runtime to load the ONNX model
#     ort_session = ort.InferenceSession(best_model_path)
# else:
#     raise FileNotFoundError(f"Best model not found at {best_model_path}")

# # Evaluate the model
# test_loss = 0.0
# test_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# test_accuracy.reset_state()

# y_true = []
# y_pred = []
# y_prob = []

# for x_test, y_test in test_dataset:
#     # Convert input to numpy for ONNX Runtime
#     x_test_np = x_test.numpy()
#     # Run inference using ONNX Runtime
#     test_logits = ort_session.run(None, {ort_session.get_inputs()[0].name: x_test_np})[0]
#     test_loss += loss_fn(y_test, test_logits).numpy()
#     test_accuracy.update_state(y_test, test_logits)

#     # Collect true labels, predicted labels, and probabilities
#     y_true.extend(y_test.numpy())
#     y_prob.extend(tf.nn.softmax(test_logits).numpy() if num_classes > 2 else tf.nn.sigmoid(test_logits).numpy())
#     y_pred.extend(tf.argmax(test_logits, axis=1).numpy() if num_classes > 2 else (test_logits > 0.5).astype(int))

# test_loss /= len(test_dataset)
# test_acc = test_accuracy.result().numpy()

# print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

# # Convert probabilities to a flat list for binary classification
# if num_classes == 2:
#     y_prob = [p[0] for p in y_prob]

# # Visualize metrics
# if num_classes == 2:
#     # Binary classification
#     plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix (Binary Classification)")
#     plot_roc_curve(y_true, y_prob, title="ROC Curve (Binary Classification)")
#     print_metrics(y_true, y_pred, y_prob)
# else:
#     # Multiclass classification
#     classes = list(range(num_classes))  # Replace with actual class names if available
#     plot_confusion_matrix_multiclass(y_true, y_pred, classes, title="Confusion Matrix (Multiclass Classification)")
#     plot_roc_curve_multiclass(y_true, np.array(y_prob), classes, title="ROC Curve (Multiclass Classification)")
#     print_metrics_multiclass(y_true, y_pred, y_prob, classes)

In [23]:
# ... (your imports, class definitions, and VGGComplexNet definition) ...

activations = ['modrelu', 'crelu', 'zrelu', 'cart_relu']
# crelu only
# activations = ['crelu']
batch_size = 16
epochs = 200
n_downsamplings = 3
n_blocks = 0
num_rx_tx = 4
input_shape = (192, 14, 2)
dim = 4

# Dataset assignment
train_dataset = train_dataset_mimo_channel
valid_dataset = test_dataset_mimo_channel  # Use your actual validation set if available
test_dataset = test_dataset_mimo_channel
class ComplexAverageSparseCategoricalCrossentropy(SparseCategoricalCrossentropy):
    # def call(self, y_true, y_pred):
    #     # If y_pred is complex, convert to real
    #     if y_pred.dtype.is_complex:
    #         y_pred = (tf.math.real(y_pred) + tf.math.imag(y_pred)) / 2
    #     return super().call(y_true, y_pred)
    def call(self, y_true, y_pred):
        real_loss = sparse_categorical_crossentropy(y_true, tf.math.real(y_pred), from_logits=True)
        if y_pred.dtype.is_complex:
            imag_loss = sparse_categorical_crossentropy(y_true, tf.math.imag(y_pred), from_logits=True)
        else:
            imag_loss = real_loss
        return (real_loss + imag_loss) / 2.
num_classes = np.unique([y for _, y in train_dataset]).shape[0]  # Adjust if needed
print(f"Number of classes: {num_classes}")
for activation in activations:
    print(f"\n=== Training with activation: {activation} ===\n")
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(f"cvnn_channel_logs/{activation}", timestamp)
    save_dir = os.path.join(f"cvnn_channel_models/{activation}", timestamp)
    os.makedirs(log_dir, exist_ok=True)
    os.makedirs(save_dir, exist_ok=True)
    writer = tf.summary.create_file_writer(log_dir)

    def VGGComplexNet(input_shape=(192, 14, 2),
                     output_channels=4,
                     dim=16,
                     n_downsamplings=3,
                     n_blocks=0,
                     num_classes=2,
                     num_rx_tx=4,
                     batch_size=16,
                     norm='instance_norm'):
        inputs = complex_layers.complex_input(shape=input_shape, batch_size=batch_size)
        h = complex_layers.ComplexConv2D(dim, kernel_size=3, strides=(1,1), dilation_rate=1, padding='same', activation=activation)(inputs)
        for i in range(n_downsamplings):
            dim *= 2
            if h.shape[2] % 2 == 0:
                h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='same')(h)
            else:
                h = complex_layers.ComplexMaxPooling2D(pool_size=(2, 2), strides=(2, 1), padding='same')(h)
            value = 2 if i <= 0 else 3
            for j in range(min(value, 3)):
                h = complex_layers.ComplexConv2D(dim, kernel_size=3, strides=(1,1), dilation_rate=1, padding='same', activation=activation)(h)
        h = complex_layers.ComplexFlatten()(h)
        h = complex_layers.ComplexDense(512, activation=activation)(h)
        h = complex_layers.ComplexDense(256, activation=activation)(h)
        h = complex_layers.ComplexDense(num_classes, activation='convert_to_real_with_abs')(h)
        model = tf.keras.Model(inputs=inputs, outputs=h)
        return model

    model = VGGComplexNet(
        batch_size=batch_size,
        input_shape=input_shape,
        num_rx_tx=num_rx_tx,
        num_classes=num_classes,
        dim=dim,
        n_downsamplings=n_downsamplings,
        n_blocks=n_blocks,
        norm='instance_norm'
    )
    model = SimpleComplexNet(
        input_shape=input_shape,
        num_classes=num_classes,
        batch_size=batch_size,
        activation=activation
    )
    loss_fn = ComplexAverageSparseCategoricalCrossentropy()
    optimizer = tf.keras.optimizers.SGD(learning_rate=0.0001, momentum=0.9)
    train_accuracy = ComplexSparseCategoricalAccuracy()
    val_accuracy = ComplexSparseCategoricalAccuracy()
    test_accuracy = ComplexSparseCategoricalAccuracy()

    best_loss = float('inf')
    patience = 10

    @tf.function
    def train_step(x_batch, y_batch):
        with tf.GradientTape() as tape:
            logits = model(x_batch, training=True)
            loss = loss_fn(y_batch, logits)
            # tf.print("Loss:", loss)
            # tf.print("Logits:", logits)
        gradients = tape.gradient(loss, model.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model.trainable_variables))
        train_accuracy.update_state(y_batch, logits)
        return loss

    @tf.function
    def test_step(x_batch, y_batch):
        logits = model(x_batch, training=False)
        loss = loss_fn(y_batch, logits)
        # tf.print("Logits:", logits)
        # tf.print("y_batch:", y_batch)
        test_accuracy.update_state(y_batch, logits)
        return loss

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")
        epoch_loss = 0.0
        train_accuracy.reset_state()

        for x_batch, y_batch in train_dataset:
            loss = train_step(x_batch, y_batch)
            epoch_loss += loss.numpy()
        epoch_loss /= len(train_dataset)
        train_acc = train_accuracy.result().numpy()

        with writer.as_default():
            tf.summary.scalar("training loss", epoch_loss, step=epoch)
            tf.summary.scalar("training accuracy", train_acc, step=epoch)

        print(f"Epoch {epoch + 1} Loss: {epoch_loss:.4f}, Accuracy: {train_acc:.4f}")

        # Validation loop
        test_loss = 0.0
        test_accuracy.reset_state()
        for x_test, y_test in test_dataset:
            loss = test_step(x_test, y_test)
            test_loss += loss.numpy()
        test_loss /= len(test_dataset)
        test_acc = test_accuracy.result().numpy()

        with writer.as_default():
            tf.summary.scalar("test loss", test_loss, step=epoch)
            tf.summary.scalar("test accuracy", test_acc, step=epoch)

        print(f"Validation Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}")

        # Save the Keras model at each epoch
        keras_model_path = os.path.join(save_dir, f"model_epoch_{epoch + 1:02d}.keras")
        model.save(keras_model_path)
        print(f"Model for epoch {epoch + 1} saved at {keras_model_path}")

        # Save the best model based on validation loss
        if test_loss < best_loss or epoch == 0:
            best_loss = test_loss
            best_keras_model_path = os.path.join(save_dir, "best_model.keras")
            model.save(best_keras_model_path)
            print("Best model saved in Keras format based on test loss")
            patience = 10
        else:
            patience -= 1
            if patience == 0:
                print("Early stopping triggered")
                break

    # Load the best model
    best_model_path = os.path.join(save_dir, "best_model.keras")
    if os.path.exists(best_model_path):
        print(f"Loading best model from {best_model_path}")
        model = tf.keras.models.load_model(best_model_path, compile=False)
    else:
        raise FileNotFoundError(f"Best model not found at {best_model_path}")

    # Evaluate the model
    test_loss = 0.0
    test_accuracy.reset_state()
    y_true = []
    y_pred = []
    y_prob = []

    for x_test, y_test in test_dataset:
        logits = model(x_test, training=False)
        loss = loss_fn(y_test, logits)
        test_loss += loss.numpy()
        test_accuracy.update_state(y_test, logits)
        y_true.extend(y_test.numpy())
        if num_classes > 2:
            y_prob.extend(tf.nn.softmax(logits).numpy())
            y_pred.extend(tf.argmax(logits, axis=1).numpy())
        else:
            y_prob.extend(tf.nn.sigmoid(logits).numpy())
            y_pred.extend((logits > 0.5).numpy().astype(int))

    test_loss /= len(test_dataset)
    test_acc = test_accuracy.result().numpy()
    print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

Number of classes: 10

=== Training with activation: modrelu ===

Epoch 1/200
Epoch 1 Loss: 2.3502, Accuracy: 0.2916
Validation Loss: 2.0707, Accuracy: 0.3565
Model for epoch 1 saved at cvnn_channel_models/modrelu/20250620-135406/model_epoch_01.keras
Best model saved in Keras format based on test loss
Epoch 2/200
Epoch 2 Loss: 0.9275, Accuracy: 0.6995
Validation Loss: 2.0900, Accuracy: 0.3595
Model for epoch 2 saved at cvnn_channel_models/modrelu/20250620-135406/model_epoch_02.keras
Epoch 3/200
Epoch 3 Loss: 0.4508, Accuracy: 0.8913
Validation Loss: 2.1570, Accuracy: 0.3631
Model for epoch 3 saved at cvnn_channel_models/modrelu/20250620-135406/model_epoch_03.keras
Epoch 4/200
Epoch 4 Loss: 0.2496, Accuracy: 0.9706
Validation Loss: 2.2302, Accuracy: 0.3586
Model for epoch 4 saved at cvnn_channel_models/modrelu/20250620-135406/model_epoch_04.keras
Epoch 5/200
Epoch 5 Loss: 0.1577, Accuracy: 0.9949
Validation Loss: 2.2836, Accuracy: 0.3613
Model for epoch 5 saved at cvnn_channel_models/mo

ValueError: Unknown layer: 'ComplexInput'. Please ensure you are using a `keras.utils.custom_object_scope` and that this object is included in the scope. See https://www.tensorflow.org/guide/keras/save_and_serialize#registering_the_custom_object for details.

In [ ]:
# # TensorBoard writer
# writer = tf.summary.create_file_writer(channel_log_dir)

# # Define the loss function
# loss_fn = tf.keras.losses.BinaryCrossentropy() if num_classes == 2 else tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)
# # print the loss function
# # Define the optimizer
# optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002)

# # Define metrics
# train_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# val_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# # Training loop
# epochs = 100  # Set the number of epochs
# best_loss = float('inf')  # Track the best validation loss

# train_dataset = train_channel_dataset
# valid_dataset = test_channel_dataset
# test_dataset = test_channel_dataset
# @tf.function
# def train_step(x_batch, y_batch):
#     with tf.GradientTape() as tape:
#         logits = model(x_batch, training=True)
#         loss = loss_fn(y_batch, logits)
#     gradients = tape.gradient(loss, model.trainable_variables)
#     optimizer.apply_gradients(zip(gradients, model.trainable_variables))
#     train_accuracy.update_state(y_batch, logits)
#     return loss

# @tf.function
# def val_step(x_batch, y_batch):
#     logits = model(x_batch, training=False)
#     loss = loss_fn(y_batch, logits)
#     val_accuracy.update_state(y_batch, logits)
#     return loss


In [ ]:
# # Training loop
# for epoch in range(epochs):
#     print(f"Epoch {epoch + 1}/{epochs}")
#     epoch_loss = 0.0
#     train_accuracy.reset_state()

#     for x_batch, y_batch in train_dataset:
#         loss = train_step(x_batch, y_batch)
#         epoch_loss += loss.numpy()
    
#     epoch_loss /= len(train_dataset)
#     train_acc = train_accuracy.result().numpy()

#     # Log training metrics to TensorBoard
#     with writer.as_default():
#         tf.summary.scalar("training loss", epoch_loss, step=epoch)
#         tf.summary.scalar("training accuracy", train_acc, step=epoch)

#     print(f"Epoch {epoch + 1} Loss: {epoch_loss:.4f}, Accuracy: {train_acc:.4f}")

#     # Validation loop
#     val_loss = 0.0
#     val_accuracy.reset_state()
#     for x_val, y_val in valid_dataset:
#         loss = val_step(x_val, y_val)
#         val_loss += loss.numpy()

#     val_loss /= len(valid_dataset)
#     val_acc = val_accuracy.result().numpy()

#     # Log validation metrics to TensorBoard
#     with writer.as_default():
#         tf.summary.scalar("validation loss", val_loss, step=epoch)
#         tf.summary.scalar("validation accuracy", val_acc, step=epoch)

#     print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")

#     # Save the ONNX model at each epoch
#     onnx_model_path = os.path.join(channel_save_dir, f"model_epoch_{epoch + 1:02d}.onnx")
#     spec = (tf.TensorSpec((None,) + input_shape, tf.float32),)
#     onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=11)
#     with open(onnx_model_path, "wb") as f:
#         f.write(onnx_model.SerializeToString())
#     print(f"Model for epoch {epoch + 1} converted to ONNX and saved at {onnx_model_path}")

#     # Save the best model based on validation loss
    
#     if val_loss < best_loss or epoch == 0:
#         best_loss = val_loss
#         best_onnx_model_path = os.path.join(channel_save_dir, "best_model.onnx")
#         with open(best_onnx_model_path, "wb") as f:
#             f.write(onnx_model.SerializeToString())
#         print("Best model saved in ONNX format based on validation loss")


In [ ]:
# # Load the best model
# best_model_path = os.path.join(channel_save_dir, "best_model.onnx")
# if os.path.exists(best_model_path):
#     print(f"Loading best model from {best_model_path}")
#     # Use ONNX Runtime to load the ONNX model
#     ort_session = ort.InferenceSession(best_model_path)
# else:
#     raise FileNotFoundError(f"Best model not found at {best_model_path}")

# # Evaluate the model
# test_loss = 0.0
# test_accuracy = BinaryAccuracy() if num_classes == 2 else SparseCategoricalAccuracy()
# test_accuracy.reset_state()

# y_true = []
# y_pred = []
# y_prob = []

# for x_test, y_test in test_dataset:
#     # Convert input to numpy for ONNX Runtime
#     x_test_np = x_test.numpy()
#     # Run inference using ONNX Runtime
#     test_logits = ort_session.run(None, {ort_session.get_inputs()[0].name: x_test_np})[0]
#     test_loss += loss_fn(y_test, test_logits).numpy()
#     test_accuracy.update_state(y_test, test_logits)

#     # Collect true labels, predicted labels, and probabilities
#     y_true.extend(y_test.numpy())
#     y_prob.extend(tf.nn.softmax(test_logits).numpy() if num_classes > 2 else tf.nn.sigmoid(test_logits).numpy())
#     y_pred.extend(tf.argmax(test_logits, axis=1).numpy() if num_classes > 2 else (test_logits > 0.5).astype(int))

# test_loss /= len(test_dataset)
# test_acc = test_accuracy.result().numpy()

# print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_acc:.4f}")

# # Convert probabilities to a flat list for binary classification
# if num_classes == 2:
#     y_prob = [p[0] for p in y_prob]

# # Visualize metrics
# if num_classes == 2:
#     # Binary classification
#     plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix (Binary Classification)")
#     plot_roc_curve(y_true, y_prob, title="ROC Curve (Binary Classification)")
#     print_metrics(y_true, y_pred, y_prob)
# else:
#     # Multiclass classification
#     classes = list(range(num_classes))  # Replace with actual class names if available
#     plot_confusion_matrix_multiclass(y_true, y_pred, classes, title="Confusion Matrix (Multiclass Classification)")
#     plot_roc_curve_multiclass(y_true, np.array(y_prob), classes, title="ROC Curve (Multiclass Classification)")
#     print_metrics_multiclass(y_true, y_pred, y_prob)
    

